In [ ]:
# Mount Google Drive and initialize Decision Tree experiment paths

from google.colab import drive
from pathlib import Path

import json
import random
import time
import hashlib
import joblib
import platform
import sys
import gc
import warnings

import numpy as np
import pandas as pd
import scipy
import scipy.sparse as sp
import sklearn
import matplotlib.pyplot as plt
import seaborn as sns

from datetime import datetime, timezone
from sklearn.tree import DecisionTreeClassifier
from sklearn.multiclass import OneVsRestClassifier

drive.mount('/content/drive')

PROJECT_ROOT = Path(
    '/content/drive/MyDrive/NLP04_Toxic_Comment_Classification'
)

CONFIG_DIR = PROJECT_ROOT / '00_config'
PROCESSED_DIR = PROJECT_ROOT / '01_data' / 'processed'
SPLITS_DIR = PROJECT_ROOT / '01_data' / 'splits'
FEATURES_DIR = PROJECT_ROOT / '01_data' / 'features'
SRC_DIR = PROJECT_ROOT / '03_src'

CLASSICAL_DIR = PROJECT_ROOT / '02_notebooks' / '03_classical_ml'
NOTEBOOK_DIR = CLASSICAL_DIR / '04_decision_tree'

RESULTS_DIR = NOTEBOOK_DIR / 'results'
TABLES_DIR = RESULTS_DIR / 'tables'
FIGURES_DIR = RESULTS_DIR / 'figures'
PREDICTIONS_DIR = RESULTS_DIR / 'predictions'
LOGS_DIR = RESULTS_DIR / 'logs'
RUN_CONFIGS_DIR = RESULTS_DIR / 'configs'

MODELS_DIR = (
    PROJECT_ROOT / '04_models' / 'classical' / 'decision_tree'
)

REGISTRY_PATH = CONFIG_DIR / 'experiment_registry.csv'

assert NOTEBOOK_DIR.is_dir(), "Notebook directory not found"

for folder in [
    TABLES_DIR,
    FIGURES_DIR,
    PREDICTIONS_DIR,
    LOGS_DIR,
    RUN_CONFIGS_DIR,
    MODELS_DIR
]:
    folder.mkdir(parents=True, exist_ok=True)


# Load the shared project configuration

with open(CONFIG_DIR / 'project_config.json', 'r') as file:
    config = json.load(file)

LABELS = config['labels']
SEED = config['seed']
FEATURE_BUDGET = config['classical_features']['feature_budget']

MODEL_NAME = 'Decision Tree'
REPRESENTATION = 'word_tfidf'
RUN_ID = 'DT_WORD_BASELINE_S42'
SCORE_TYPE = 'positive_class_probability'

DEFAULT_THRESHOLD = float(
    config['evaluation']['default_probability_threshold']
)

random.seed(SEED)
np.random.seed(SEED)

assert len(LABELS) == 6
assert DEFAULT_THRESHOLD == 0.5
assert FEATURE_BUDGET == 50000
assert config['classical_features']['status'] == 'verified'

assert REPRESENTATION in (
    config['classical_features']['representations']
)

assert REGISTRY_PATH.is_file()


# Reuse the shared evaluation module

sys.path.insert(0, str(SRC_DIR))

import evaluation_utils

from evaluation_utils import (
    apply_thresholds,
    evaluate_multilabel,
    optimize_thresholds
)

assert Path(evaluation_utils.__file__).resolve() == (
    SRC_DIR / 'evaluation_utils.py'
).resolve()


def calculate_sha256(path):

    sha256 = hashlib.sha256()

    with open(path, 'rb') as file:

        for chunk in iter(
            lambda: file.read(1024 * 1024), b''
        ):
            sha256.update(chunk)

    return sha256.hexdigest()


evaluation_hash = calculate_sha256(
    SRC_DIR / 'evaluation_utils.py'
)

feature_manifest_hash = calculate_sha256(
    FEATURES_DIR / 'feature_artifact_manifest.csv'
)


# Verify consistency with all previously completed experiments

registry_df = pd.read_csv(
    REGISTRY_PATH,
    dtype=str,
    keep_default_na=False
)

assert registry_df['run_id'].is_unique

required_previous_runs = [
    'NB_WORD_BASELINE_S42',
    'LR_WORD_BASELINE_S42',
    'LR_CHAR_BASELINE_S42',
    'LR_COMBINED_BASELINE_S42',
    'LR_COMBINED_BALANCED_S42',
    'SVM_WORD_BASELINE_S42',
    'SVM_CHAR_BASELINE_S42',
    'SVM_COMBINED_BASELINE_S42',
    'SVM_COMBINED_BALANCED_S42'
]

assert set(required_previous_runs).issubset(
    set(registry_df['run_id'])
)

previous_run_configs = {}
consistency_records = []

for run_id in required_previous_runs:

    record = registry_df.loc[
        registry_df['run_id'] == run_id
    ].iloc[0]

    assert record['status'] == 'Completed'

    config_path = PROJECT_ROOT / record['config_path']

    assert config_path.is_file()

    with open(config_path, 'r') as file:
        run_config = json.load(file)

    assert run_config['labels'] == LABELS
    assert run_config['seed'] == SEED

    assert (
        run_config['evaluation_module_sha256']
        == evaluation_hash
    )

    assert (
        run_config['feature_manifest_sha256']
        == feature_manifest_hash
    )

    assert not run_config['official_test_used']

    previous_run_configs[run_id] = run_config

    consistency_records.append({
        'Run ID': run_id,
        'Label Order Verified': True,
        'Seed Verified': True,
        'Evaluation Module Verified': True,
        'Feature Manifest Verified': True,
        'Official Test Used': False,
        'Status': 'Verified'
    })

previous_run_verification_df = pd.DataFrame(
    consistency_records
)

print("Previous experiment consistency verification")

display(previous_run_verification_df)

previous_run_verification_df.to_csv(
    TABLES_DIR / 'previous_run_consistency.csv',
    index=False
)


# Verify common training and validation fingerprints

training_hashes = {
    run_config['training_dataset_sha256']
    for run_config in previous_run_configs.values()
}

validation_hashes = {
    run_config['validation_dataset_sha256']
    for run_config in previous_run_configs.values()
}

assert len(training_hashes) == 1
assert len(validation_hashes) == 1

EXPECTED_TRAINING_HASH = next(iter(training_hashes))
EXPECTED_VALIDATION_HASH = next(iter(validation_hashes))


# Verify evaluation support for repeated probability scores

toy_labels = ['example_a', 'example_b']

toy_targets = np.array([
    [0, 0],
    [1, 0],
    [0, 1],
    [1, 1],
    [1, 0],
    [0, 1]
], dtype=np.int8)

toy_scores = np.array([
    [0.0, 0.0],
    [0.4, 0.2],
    [0.0, 0.8],
    [0.4, 0.8],
    [0.4, 0.2],
    [0.0, 0.8]
], dtype=np.float64)

toy_default_metrics, toy_default_per_label = (
    evaluate_multilabel(
        toy_targets,
        toy_scores,
        toy_labels,
        thresholds=DEFAULT_THRESHOLD
    )
)

toy_threshold_df = optimize_thresholds(
    toy_targets,
    toy_scores,
    toy_labels
)

toy_thresholds = toy_threshold_df[
    'Optimal Threshold'
].to_numpy(dtype=np.float64)

assert np.allclose(
    toy_thresholds,
    [0.4, 0.8]
)

toy_optimized_metrics, toy_optimized_per_label = (
    evaluate_multilabel(
        toy_targets,
        toy_scores,
        toy_labels,
        thresholds=toy_thresholds
    )
)

assert np.isclose(
    toy_optimized_metrics['Macro F1'],
    1.0
)

assert np.array_equal(
    apply_thresholds(toy_scores, toy_thresholds),
    toy_targets
)

for metric in [
    'Macro AP',
    'Micro AP',
    'Macro ROC-AUC',
    'Micro ROC-AUC'
]:

    assert np.isclose(
        toy_default_metrics[metric],
        toy_optimized_metrics[metric]
    )

probability_verification_df = pd.DataFrame({
    'Label': toy_labels,
    'Default Threshold': DEFAULT_THRESHOLD,
    'Optimal Threshold': toy_thresholds,
    'Default F1':
        toy_default_per_label['F1'].to_numpy(),
    'Optimized F1':
        toy_optimized_per_label['F1'].to_numpy(),
    'Repeated Scores Supported': True,
    'Verification': 'Passed'
})

print("\nProbability score evaluation verification")

display(probability_verification_df.round(6))

probability_verification_df.to_csv(
    TABLES_DIR / 'probability_score_evaluation_verification.csv',
    index=False
)


# Configure consistent visualization settings

sns.set_theme(style='whitegrid')

LABEL_COLORS = {
    'toxic': '#4C78A8',
    'severe_toxic': '#F58518',
    'obscene': '#54A24B',
    'threat': '#E45756',
    'insult': '#B279A2',
    'identity_hate': '#72B7B2'
}

MODEL_COLORS = {
    'Multinomial Naive Bayes': '#94A3B8',
    'Logistic Regression': '#4C78A8',
    'Linear SVM': '#54A24B',
    'Decision Tree': '#F58518'
}


# Record current environment versions

environment_df = pd.DataFrame([
    {'Package': 'Python', 'Version': platform.python_version()},
    {'Package': 'NumPy', 'Version': np.__version__},
    {'Package': 'Pandas', 'Version': pd.__version__},
    {'Package': 'SciPy', 'Version': scipy.__version__},
    {'Package': 'Scikit-learn', 'Version': sklearn.__version__},
    {'Package': 'Joblib', 'Version': joblib.__version__}
])

environment_df.to_csv(
    TABLES_DIR / 'environment_versions.csv',
    index=False
)

print("\nCurrent environment versions")

display(environment_df)


# Compare environment with the feature extraction notebook

feature_environment_path = (
    CLASSICAL_DIR / '00_feature_extraction'
    / 'results' / 'tables' / 'environment_versions.csv'
)

assert feature_environment_path.is_file()

feature_environment_df = pd.read_csv(
    feature_environment_path
)

environment_comparison_df = environment_df.merge(
    feature_environment_df,
    on='Package',
    suffixes=('_Current', '_Feature_Extraction'),
    validate='one_to_one'
)

assert len(environment_comparison_df) == len(environment_df)

environment_comparison_df['Version Match'] = (
    environment_comparison_df['Version_Current']
    == environment_comparison_df['Version_Feature_Extraction']
)

print("\nEnvironment compatibility")

display(environment_comparison_df)

environment_comparison_df.to_csv(
    TABLES_DIR / 'environment_compatibility.csv',
    index=False
)

assert environment_comparison_df['Version Match'].all(), (
    "Environment version mismatch requires review"
)


# Summarize project initialization

print("\nDecision Tree initialization summary")

print("Notebook:", NOTEBOOK_DIR.name)
print("Model:", MODEL_NAME)
print("Planned run:", RUN_ID)
print("Feature representation:", REPRESENTATION)
print("Feature budget:", FEATURE_BUDGET)
print("Target labels:", LABELS)
print("Random seed:", SEED)
print("Score type:", SCORE_TYPE)
print("Default threshold:", DEFAULT_THRESHOLD)
print("Results directory:", RESULTS_DIR)
print("Model directory:", MODELS_DIR)

print("Previous experiments verified:",
      len(previous_run_verification_df))

print("Matching environment packages:",
      environment_comparison_df['Version Match'].sum(),
      "/", len(environment_df))

print("Shared dataset fingerprints: Consistent")
print("Shared evaluation module: Verified")
print("Repeated probability scores: Supported")
print("Feature artifact manifest: Verified")
print("Model training performed: No")
print("Official test dataset: Not loaded")

print("\nDecision Tree environment initialized successfully.")

In [ ]:
# Load the verified dataset and feature manifests

dataset_manifest = pd.read_csv(
    SPLITS_DIR / 'prepared_dataset_manifest.csv'
)

feature_manifest = pd.read_csv(
    FEATURES_DIR / 'feature_artifact_manifest.csv'
)

datasets = {}
dataset_records = {}
verification_records = []

# Verify training and validation datasets

for partition, source_partition, expected_hash in [
    ('train', 'train_representative', EXPECTED_TRAINING_HASH),
    ('validation', 'validation', EXPECTED_VALIDATION_HASH)
]:

    matching_records = dataset_manifest.loc[
        dataset_manifest['Partition'] == source_partition
    ]

    assert len(matching_records) == 1

    record = matching_records.iloc[0]

    dataset_path = PROJECT_ROOT / record['Dataset Path']

    assert dataset_path.is_file()

    dataset_hash = calculate_sha256(dataset_path)

    assert dataset_hash == record['SHA256']
    assert dataset_hash == expected_hash

    df = pd.read_parquet(dataset_path)

    assert len(df) == record['Rows']
    assert df['id'].is_unique
    assert df['group_id'].notna().all()
    assert df['comment_text'].notna().all()
    assert df[LABELS].isin([0, 1]).all().all()

    # Verify feature row alignment

    row_records = feature_manifest.loc[
        (feature_manifest['Artifact Type'] == 'Row Manifest')
        & (feature_manifest['Partition'] == partition)
    ]

    assert len(row_records) == 1

    row_record = row_records.iloc[0]

    row_path = PROJECT_ROOT / row_record['Path']

    assert calculate_sha256(row_path) == row_record['SHA256']

    assert (
        row_record['Source Dataset SHA256']
        == dataset_hash
    )

    row_manifest = pd.read_csv(
        row_path,
        dtype={'id': str}
    )

    assert len(row_manifest) == len(df)

    assert (
        row_manifest['id'].tolist()
        == df['id'].tolist()
    )

    assert (
        row_manifest['group_id'].tolist()
        == df['group_id'].tolist()
    )

    assert np.array_equal(
        row_manifest['row_index'].to_numpy(),
        np.arange(len(df))
    )

    datasets[partition] = df
    dataset_records[partition] = record

    verification_records.append({
        'Partition': partition,
        'Comments': len(df),
        'Unique IDs': df['id'].nunique(),
        'Dataset Hash Verified': True,
        'Previous Experiments Match': True,
        'Row Manifest Verified': True,
        'Row Alignment Verified': True,
        'Valid Labels': True
    })

train_df = datasets['train']
val_df = datasets['validation']

y_train = train_df[LABELS].to_numpy(dtype=np.int8)
y_val = val_df[LABELS].to_numpy(dtype=np.int8)

assert y_train.shape == (59997, len(LABELS))
assert y_val.shape == (31911, len(LABELS))

assert set(train_df['id']).isdisjoint(
    val_df['id']
)

assert set(train_df['group_id']).isdisjoint(
    val_df['group_id']
)

assert (y_train.sum(axis=0) > 0).all()
assert (y_val.sum(axis=0) > 0).all()

assert (y_train.sum(axis=0) < len(y_train)).all()
assert (y_val.sum(axis=0) < len(y_val)).all()

dataset_verification_df = pd.DataFrame(
    verification_records
)

print("Dataset integrity verification")

display(dataset_verification_df)

dataset_verification_df.to_csv(
    TABLES_DIR / 'dataset_verification.csv',
    index=False
)


# Define the verified word-feature loader

def load_verified_word_features():

    matrices = {}
    records = []

    for partition in ['train', 'validation']:

        matching_records = feature_manifest.loc[
            (feature_manifest['Artifact Type'] == 'Sparse Matrix')
            & (feature_manifest['Representation'] == REPRESENTATION)
            & (feature_manifest['Partition'] == partition)
        ]

        assert len(matching_records) == 1

        record = matching_records.iloc[0]

        path = PROJECT_ROOT / record['Path']

        assert path.is_file()

        assert calculate_sha256(path) == record['SHA256']

        assert (
            record['Source Dataset SHA256']
            == dataset_records[partition]['SHA256']
        )

        matrix = sp.load_npz(path)

        assert matrix.format == 'csr'
        assert matrix.dtype == np.float32

        assert matrix.shape == (
            len(datasets[partition]),
            FEATURE_BUDGET
        )

        assert np.isfinite(matrix.data).all()
        assert (matrix.data >= 0).all()

        memory_bytes = (
            matrix.data.nbytes
            + matrix.indices.nbytes
            + matrix.indptr.nbytes
        )

        matrices[partition] = matrix

        records.append({
            'Representation': REPRESENTATION,
            'Partition': partition,
            'Comments': matrix.shape[0],
            'Features': matrix.shape[1],
            'Nonzero Elements': matrix.nnz,
            'Data Type': str(matrix.dtype),
            'Sparse Memory (MiB)':
                memory_bytes / (1024 ** 2),
            'Zero Feature Comments':
                int(np.sum(np.diff(matrix.indptr) == 0)),
            'Feature Hash Verified': True,
            'Source Dataset Verified': True,
            'Verification': 'Passed'
        })

    return (
        matrices['train'],
        matrices['validation'],
        pd.DataFrame(records)
    )


# Load the common word TF-IDF matrices

loading_start = time.perf_counter()

X_train, X_val, feature_verification_df = (
    load_verified_word_features()
)

feature_loading_time = (
    time.perf_counter() - loading_start
)

print("\nWord TF-IDF feature verification")

display(feature_verification_df.round(4))

feature_verification_df.to_csv(
    TABLES_DIR / 'feature_representation_verification.csv',
    index=False
)


# Verify identical feature properties across LR and SVM

comparison_columns = [
    'Representation',
    'Partition',
    'Comments',
    'Features',
    'Nonzero Elements',
    'Data Type',
    'Sparse Memory (MiB)',
    'Zero Feature Comments'
]

consistency_checks = []

for notebook in [
    '02_logistic_regression',
    '03_linear_svm'
]:

    previous_feature_df = pd.read_csv(
        CLASSICAL_DIR / notebook
        / 'results' / 'tables'
        / 'feature_representation_verification.csv'
    )

    previous_word_df = previous_feature_df.loc[
        previous_feature_df['Representation']
        == REPRESENTATION
    ]

    current_comparison = (
        feature_verification_df[comparison_columns]
        .sort_values('Partition')
        .reset_index(drop=True)
    )

    previous_comparison = (
        previous_word_df[comparison_columns]
        .sort_values('Partition')
        .reset_index(drop=True)
    )

    pd.testing.assert_frame_equal(
        current_comparison,
        previous_comparison,
        check_exact=False,
        rtol=1e-12,
        atol=1e-12
    )

    consistency_checks.append({
        'Previous Notebook': notebook,
        'Representation': REPRESENTATION,
        'Training Features Match': True,
        'Validation Features Match': True,
        'Verification': 'Passed'
    })

feature_consistency_df = pd.DataFrame(
    consistency_checks
)

print("\nPrevious experiment feature consistency")

display(feature_consistency_df)

feature_consistency_df.to_csv(
    TABLES_DIR / 'previous_feature_consistency.csv',
    index=False
)


# Record label support and class imbalance

label_support_df = pd.DataFrame({
    'Label': LABELS,
    'Training Positives': y_train.sum(axis=0),
    'Training Negatives': len(y_train) - y_train.sum(axis=0),
    'Validation Positives': y_val.sum(axis=0),
    'Validation Negatives': len(y_val) - y_val.sum(axis=0)
})

label_support_df['Training Imbalance Ratio'] = (
    label_support_df['Training Negatives']
    / label_support_df['Training Positives']
)

label_support_df['Training Prevalence (%)'] = (
    100 * label_support_df['Training Positives']
    / len(y_train)
)

label_support_df['Validation Prevalence (%)'] = (
    100 * label_support_df['Validation Positives']
    / len(y_val)
)

print("\nLabel support and class imbalance")

display(label_support_df.round(6))

label_support_df.to_csv(
    TABLES_DIR / 'label_support_and_imbalance.csv',
    index=False
)


# Calculate the sparse feature storage requirements

training_memory_mib = (
    X_train.data.nbytes
    + X_train.indices.nbytes
    + X_train.indptr.nbytes
) / (1024 ** 2)

validation_memory_mib = (
    X_val.data.nbytes
    + X_val.indices.nbytes
    + X_val.indptr.nbytes
) / (1024 ** 2)

print("\nDataset and feature verification summary")

print("Training label shape:", y_train.shape)
print("Validation label shape:", y_val.shape)

print("Training feature shape:", X_train.shape)
print("Validation feature shape:", X_val.shape)

print("Training nonzero elements:", X_train.nnz)
print("Validation nonzero elements:", X_val.nnz)

print("Training sparse memory (MiB):",
      round(training_memory_mib, 4))

print("Validation sparse memory (MiB):",
      round(validation_memory_mib, 4))

print("Total sparse matrix memory (MiB):",
      round(training_memory_mib + validation_memory_mib, 4))

print("Feature loading time (s):",
      round(feature_loading_time, 4))

print("Sparse matrices verified:", len(feature_verification_df))
print("Previous LR and SVM feature consistency: Verified")
print("Training-validation ID overlap: 0")
print("Training-validation group overlap: 0")
print("Feature matrices retained for training: Yes")
print("Dense matrix conversion: No")
print("Model training performed: No")
print("Official test dataset: Not loaded")

gc.collect()

print("\nDecision Tree dataset and feature verification completed successfully.")

In [ ]:
# Define the resource-controlled Decision Tree baseline

DT_PARAMS = {
    'criterion': 'gini',
    'splitter': 'best',
    'max_depth': 30,
    'min_samples_split': 10,
    'min_samples_leaf': 5,
    'min_weight_fraction_leaf': 0.0,
    'max_features': None,
    'random_state': SEED,
    'max_leaf_nodes': 512,
    'min_impurity_decrease': 0.0,
    'class_weight': None,
    'ccp_alpha': 0.0
}

OVR_PARAMS = {
    'n_jobs': 1
}


# Define the baseline experiment

experiment_spec = {
    'run_id': RUN_ID,
    'representation': REPRESENTATION,
    'class_weight': None,
    'purpose': 'Resource-controlled nonlinear classical baseline'
}


# Construct the multi-label classifier

def build_dt_model():

    estimator = DecisionTreeClassifier(
        **DT_PARAMS
    )

    model = OneVsRestClassifier(
        estimator,
        **OVR_PARAMS
    )

    return model


dt_model = build_dt_model()

assert isinstance(
    dt_model.estimator,
    DecisionTreeClassifier
)

assert dt_model.n_jobs == 1

for parameter, value in DT_PARAMS.items():

    assert (
        dt_model.estimator.get_params()[parameter]
        == value
    )

assert hasattr(dt_model, 'predict_proba')
assert DT_PARAMS['class_weight'] is None


# Record hyperparameter rationale before model training

hyperparameter_rationale = [
    {
        'Parameter': 'criterion',
        'Value': 'gini',
        'Justification': (
            'Standard impurity criterion for binary '
            'classification with low computational overhead.'
        )
    },
    {
        'Parameter': 'splitter',
        'Value': 'best',
        'Justification': (
            'Search for the best available split at each '
            'node rather than introducing random splits.'
        )
    },
    {
        'Parameter': 'max_depth',
        'Value': '30',
        'Justification': (
            'Restrict excessive tree depth while allowing '
            'nonlinear feature interactions.'
        )
    },
    {
        'Parameter': 'min_samples_split',
        'Value': '10',
        'Justification': (
            'Prevent further splitting of nodes containing '
            'fewer than ten samples.'
        )
    },
    {
        'Parameter': 'min_samples_leaf',
        'Value': '5',
        'Justification': (
            'Reduce extremely small terminal leaves and '
            'unstable leaf probability estimates while '
            'retaining flexibility for rare classes.'
        )
    },
    {
        'Parameter': 'max_leaf_nodes',
        'Value': '512',
        'Justification': (
            'Place an explicit upper bound on tree size '
            'and computational complexity using best-first '
            'tree growth.'
        )
    },
    {
        'Parameter': 'max_features',
        'Value': 'None',
        'Justification': (
            'Allow access to the complete common word '
            'feature representation when searching for splits.'
        )
    },
    {
        'Parameter': 'class_weight',
        'Value': 'None',
        'Justification': (
            'Maintain an unweighted common-feature baseline '
            'for comparison with Naive Bayes, Logistic '
            'Regression and Linear SVM.'
        )
    },
    {
        'Parameter': 'ccp_alpha',
        'Value': '0.0',
        'Justification': (
            'Avoid additional cost-complexity pruning '
            'because explicit tree growth restrictions '
            'are already imposed.'
        )
    },
    {
        'Parameter': 'random_state',
        'Value': str(SEED),
        'Justification': (
            'Control estimator randomness for reproducibility.'
        )
    },
    {
        'Parameter': 'n_jobs',
        'Value': '1',
        'Justification': (
            'Train the six One-vs-Rest estimators '
            'sequentially to limit concurrent memory usage.'
        )
    }
]

dt_hyperparameter_rationale_df = pd.DataFrame(
    hyperparameter_rationale
)

print("Decision Tree hyperparameter rationale")

display(dt_hyperparameter_rationale_df)

dt_hyperparameter_rationale_df.to_csv(
    TABLES_DIR / 'dt_hyperparameter_rationale.csv',
    index=False
)


# Record theoretical structural restrictions

maximum_leaves = DT_PARAMS['max_leaf_nodes']

maximum_nodes = (
    2 * maximum_leaves - 1
)

structural_limits_df = pd.DataFrame({
    'Label': LABELS,
    'Training Positives': y_train.sum(axis=0),
    'Training Negatives': (
        len(y_train) - y_train.sum(axis=0)
    ),
    'Maximum Depth': DT_PARAMS['max_depth'],
    'Maximum Leaves': maximum_leaves,
    'Maximum Nodes': maximum_nodes,
    'Minimum Samples per Leaf':
        DT_PARAMS['min_samples_leaf'],
    'Minimum Samples to Split':
        DT_PARAMS['min_samples_split']
})

print("\nPer-classifier structural restrictions")

display(structural_limits_df)

structural_limits_df.to_csv(
    TABLES_DIR / 'dt_structural_limits.csv',
    index=False
)


# Save the predefined experiment configuration

dt_experiment_config = {
    'notebook': NOTEBOOK_DIR.name,
    'run_id': RUN_ID,
    'model': MODEL_NAME,
    'estimator': 'DecisionTreeClassifier',
    'classification_strategy': 'One-vs-Rest',
    'labels': LABELS,
    'num_labels': len(LABELS),
    'seed': SEED,
    'sampling_strategy': 'representative_multilabel',
    'training_samples': len(y_train),
    'validation_samples': len(y_val),
    'feature_representation': REPRESENTATION,
    'feature_budget': FEATURE_BUDGET,
    'model_parameters': DT_PARAMS,
    'wrapper_parameters': OVR_PARAMS,
    'score_type': SCORE_TYPE,
    'default_threshold': DEFAULT_THRESHOLD,
    'threshold_comparison_operator': '>=',
    'threshold_optimization_metric': 'per_label_f1',
    'threshold_tie_breaking': 'highest_threshold',
    'primary_selection_metric': 'macro_average_precision',
    'secondary_selection_metric': 'optimized_macro_f1',
    'probability_calibration': False,
    'hyperparameter_selection_strategy':
        'predefined_resource_controlled_baseline',
    'hyperparameter_search_performed': False,
    'feature_representation_search_performed': False,
    'class_weighting_experiment_planned': False,
    'maximum_nodes_per_classifier': maximum_nodes,
    'maximum_nodes_across_classifiers':
        maximum_nodes * len(LABELS),
    'training_dataset_sha256': EXPECTED_TRAINING_HASH,
    'validation_dataset_sha256': EXPECTED_VALIDATION_HASH,
    'feature_manifest_sha256': feature_manifest_hash,
    'evaluation_module_sha256': evaluation_hash,
    'sklearn_version': sklearn.__version__,
    'official_test_used': False
}

DT_EXPERIMENT_CONFIG_PATH = (
    RUN_CONFIGS_DIR / 'dt_experiment_config.json'
)

with open(
    DT_EXPERIMENT_CONFIG_PATH, 'w'
) as file:

    json.dump(
        dt_experiment_config,
        file,
        indent=4
    )

with open(
    DT_EXPERIMENT_CONFIG_PATH, 'r'
) as file:

    saved_dt_config = json.load(file)

assert saved_dt_config == dt_experiment_config


# Record the planned experiment

dt_experiment_plan_df = pd.DataFrame([{
    'Run ID': RUN_ID,
    'Model': MODEL_NAME,
    'Representation': REPRESENTATION,
    'Class Weight': 'None',
    'Training Samples': len(y_train),
    'Validation Samples': len(y_val),
    'Features': FEATURE_BUDGET,
    'Binary Classifiers': len(LABELS),
    'Criterion': DT_PARAMS['criterion'],
    'Maximum Depth': DT_PARAMS['max_depth'],
    'Maximum Leaf Nodes': DT_PARAMS['max_leaf_nodes'],
    'Minimum Samples per Leaf':
        DT_PARAMS['min_samples_leaf'],
    'Default Threshold': DEFAULT_THRESHOLD,
    'Primary Selection Metric': 'Macro AP',
    'Purpose': experiment_spec['purpose'],
    'Status': 'Planned'
}])

print("\nDecision Tree experiment plan")

display(dt_experiment_plan_df.T)

dt_experiment_plan_df.to_csv(
    TABLES_DIR / 'dt_experiment_plan.csv',
    index=False
)


# Initialize result storage

dt_results = {}


# Summarize the predefined configuration

print("\nDecision Tree configuration summary")

print("Planned run:", RUN_ID)
print("Estimator:", type(dt_model.estimator).__name__)
print("Classification strategy: One-vs-Rest")
print("Binary classifiers planned:", len(LABELS))
print("Feature representation:", REPRESENTATION)
print("Training samples:", len(y_train))
print("Validation samples:", len(y_val))
print("Feature count:", FEATURE_BUDGET)

print("Criterion:", DT_PARAMS['criterion'])
print("Maximum depth:", DT_PARAMS['max_depth'])
print("Minimum samples to split:",
      DT_PARAMS['min_samples_split'])
print("Minimum samples per leaf:",
      DT_PARAMS['min_samples_leaf'])

print("Maximum leaves per classifier:", maximum_leaves)
print("Maximum nodes per classifier:", maximum_nodes)

print("Maximum nodes across all classifiers:",
      maximum_nodes * len(LABELS))

print("Class weighting:", DT_PARAMS['class_weight'])
print("Default threshold:", DEFAULT_THRESHOLD)
print("Primary selection metric: Macro AP")
print("Hyperparameter search: Not performed")
print("Configuration saved and verified: Yes")
print("Model training performed: No")
print("Official test dataset: Not loaded")

print("\nDecision Tree configuration completed successfully.")

In [ ]:
# Prepare experiment-specific artifact directories

RUN_DIR = RESULTS_DIR / RUN_ID

RUN_TABLES_DIR = RUN_DIR / 'tables'
RUN_PREDICTIONS_DIR = RUN_DIR / 'predictions'
RUN_LOGS_DIR = RUN_DIR / 'logs'

for folder in [
    RUN_TABLES_DIR,
    RUN_PREDICTIONS_DIR,
    RUN_LOGS_DIR
]:
    folder.mkdir(parents=True, exist_ok=True)

MODEL_PATH = MODELS_DIR / f'{RUN_ID}.joblib'

assert X_train.shape == (59997, FEATURE_BUDGET)
assert X_val.shape == (31911, FEATURE_BUDGET)

assert sp.issparse(X_train)
assert sp.issparse(X_val)

assert X_train.dtype == np.float32
assert X_val.dtype == np.float32

assert y_train.shape == (len(train_df), len(LABELS))
assert y_val.shape == (len(val_df), len(LABELS))


# Verify the predefined experiment configuration

with open(DT_EXPERIMENT_CONFIG_PATH, 'r') as file:
    saved_dt_config = json.load(file)

assert saved_dt_config['model_parameters'] == DT_PARAMS
assert saved_dt_config['wrapper_parameters'] == OVR_PARAMS
assert not saved_dt_config['official_test_used']

dt_model = build_dt_model()

print("Starting experiment:", RUN_ID)
print("Representation:", REPRESENTATION)
print("Class weight:", DT_PARAMS['class_weight'])
print("Training features:", X_train.shape)
print("Validation features:", X_val.shape)
print("Binary classifiers:", len(LABELS))


# Convert training features to CSC for efficient tree fitting

training_start_time = datetime.now(timezone.utc).isoformat()

training_start = time.perf_counter()

conversion_start = time.perf_counter()

X_train_csc = X_train.tocsc()
X_train_csc.sort_indices()

feature_conversion_time = (
    time.perf_counter() - conversion_start
)

assert X_train_csc.shape == X_train.shape
assert X_train_csc.nnz == X_train.nnz
assert X_train_csc.dtype == np.float32

print("\nSparse training format:", X_train_csc.format)
print("Sparse conversion time (s):",
      round(feature_conversion_time, 6))

print("\nTraining Decision Tree classifiers...")


# Train six independent binary classifiers

fit_start = time.perf_counter()

with warnings.catch_warnings(record=True) as captured_warnings:

    warnings.simplefilter('always')

    dt_model.fit(
        X_train_csc,
        y_train
    )

model_fit_time = time.perf_counter() - fit_start

training_time = time.perf_counter() - training_start

training_end_time = datetime.now(timezone.utc).isoformat()

print("Model training completed.")

print("Model fitting time (s):",
      round(model_fit_time, 6))

print("Total training time including CSC conversion (s):",
      round(training_time, 6))

del X_train_csc

gc.collect()


# Record warnings generated during fitting

training_warning_records = [
    {
        'Category': warning.category.__name__,
        'Message': str(warning.message)
    }
    for warning in captured_warnings
]

with open(
    RUN_LOGS_DIR / 'training_warnings.json',
    'w'
) as file:

    json.dump(
        training_warning_records,
        file,
        indent=4
    )

print("\nTraining warnings:",
      len(training_warning_records))

if training_warning_records:

    display(
        pd.DataFrame(training_warning_records)
    )


# Verify One-vs-Rest classifier alignment

assert dt_model.multilabel_
assert len(dt_model.estimators_) == len(LABELS)

assert np.array_equal(
    dt_model.classes_,
    np.arange(len(LABELS))
)

assert dt_model.n_features_in_ == FEATURE_BUDGET


# Inspect the structure of each fitted Decision Tree

structure_records = []

for label_index, (label, estimator) in enumerate(
    zip(LABELS, dt_model.estimators_)
):

    assert isinstance(
        estimator,
        DecisionTreeClassifier
    )

    assert np.array_equal(
        estimator.classes_,
        np.array([0, 1])
    )

    assert estimator.n_features_in_ == FEATURE_BUDGET

    for parameter, value in DT_PARAMS.items():

        assert estimator.get_params()[parameter] == value

    tree = estimator.tree_

    leaf_mask = tree.children_left == -1
    internal_mask = ~leaf_mask

    depth = estimator.get_depth()
    leaf_count = estimator.get_n_leaves()
    node_count = tree.node_count

    leaf_samples = tree.n_node_samples[leaf_mask]

    used_features = tree.feature[internal_mask]

    assert np.isfinite(tree.threshold).all()
    assert np.isfinite(tree.impurity).all()
    assert np.isfinite(tree.value).all()

    assert depth <= DT_PARAMS['max_depth']

    assert leaf_count <= DT_PARAMS['max_leaf_nodes']

    assert node_count <= maximum_nodes

    assert node_count == (2 * leaf_count - 1)

    assert leaf_mask.sum() == leaf_count

    assert leaf_samples.min() >= (
        DT_PARAMS['min_samples_leaf']
    )

    assert leaf_samples.sum() == len(y_train)

    assert tree.n_node_samples[0] == len(y_train)

    assert (
        tree.children_right[leaf_mask] == -1
    ).all()

    if internal_mask.any():

        assert (
            tree.n_node_samples[internal_mask]
            >= DT_PARAMS['min_samples_split']
        ).all()

        assert (used_features >= 0).all()
        assert (used_features < FEATURE_BUDGET).all()

    structure_records.append({
        'Label': label,
        'Training Positives':
            int(y_train[:, label_index].sum()),
        'Training Negatives':
            int(len(y_train) - y_train[:, label_index].sum()),
        'Actual Depth': depth,
        'Maximum Depth': DT_PARAMS['max_depth'],
        'Leaf Count': leaf_count,
        'Maximum Leaves': DT_PARAMS['max_leaf_nodes'],
        'Node Count': node_count,
        'Maximum Nodes': maximum_nodes,
        'Minimum Leaf Samples': int(leaf_samples.min()),
        'Maximum Leaf Samples': int(leaf_samples.max()),
        'Mean Leaf Samples': float(leaf_samples.mean()),
        'Median Leaf Samples': float(np.median(leaf_samples)),
        'Unique Split Features': int(len(np.unique(used_features))),
        'Root Impurity': float(tree.impurity[0]),
        'Depth Limit Reached': depth == DT_PARAMS['max_depth'],
        'Leaf Limit Reached':
            leaf_count == DT_PARAMS['max_leaf_nodes'],
        'Class Alignment': 'Passed',
        'Structural Verification': 'Passed'
    })

dt_structure_df = pd.DataFrame(
    structure_records
)

assert len(dt_structure_df) == len(LABELS)

assert dt_structure_df[
    'Structural Verification'
].eq('Passed').all()

print("\nDecision Tree structural verification")

display(dt_structure_df.round(6))

dt_structure_df.to_csv(
    RUN_TABLES_DIR / 'classifier_structure_summary.csv',
    index=False
)


# Summarize actual tree complexity

dt_complexity_df = pd.DataFrame([
    {
        'Property': 'Total binary classifiers',
        'Actual': len(dt_model.estimators_),
        'Configured Limit': len(LABELS)
    },
    {
        'Property': 'Maximum observed depth',
        'Actual': int(dt_structure_df['Actual Depth'].max()),
        'Configured Limit': DT_PARAMS['max_depth']
    },
    {
        'Property': 'Total leaves',
        'Actual': int(dt_structure_df['Leaf Count'].sum()),
        'Configured Limit':
            DT_PARAMS['max_leaf_nodes'] * len(LABELS)
    },
    {
        'Property': 'Total nodes',
        'Actual': int(dt_structure_df['Node Count'].sum()),
        'Configured Limit':
            maximum_nodes * len(LABELS)
    },
    {
        'Property': 'Maximum observed leaves per classifier',
        'Actual': int(dt_structure_df['Leaf Count'].max()),
        'Configured Limit': DT_PARAMS['max_leaf_nodes']
    },
    {
        'Property': 'Maximum observed nodes per classifier',
        'Actual': int(dt_structure_df['Node Count'].max()),
        'Configured Limit': maximum_nodes
    }
])

print("\nOverall model complexity")

display(dt_complexity_df)

dt_complexity_df.to_csv(
    RUN_TABLES_DIR / 'model_complexity_summary.csv',
    index=False
)


# Save the trained model

joblib.dump(
    dt_model,
    MODEL_PATH,
    compress=3
)

assert MODEL_PATH.is_file()
assert MODEL_PATH.stat().st_size > 0

model_size_mib = (
    MODEL_PATH.stat().st_size / (1024 ** 2)
)

model_hash = calculate_sha256(MODEL_PATH)


# Save training metadata for subsequent evaluation

training_log = {
    'run_id': RUN_ID,
    'model': MODEL_NAME,
    'feature_representation': REPRESENTATION,
    'classification_strategy': 'One-vs-Rest',
    'training_samples': len(y_train),
    'validation_samples': len(y_val),
    'features': FEATURE_BUDGET,
    'binary_classifiers': len(dt_model.estimators_),
    'training_start_time': training_start_time,
    'training_end_time': training_end_time,
    'feature_loading_time_seconds': float(feature_loading_time),
    'sparse_conversion_time_seconds': float(feature_conversion_time),
    'model_fit_time_seconds': float(model_fit_time),
    'training_time_seconds': float(training_time),
    'training_warnings': len(training_warning_records),
    'maximum_observed_depth':
        int(dt_structure_df['Actual Depth'].max()),
    'total_leaves':
        int(dt_structure_df['Leaf Count'].sum()),
    'total_nodes':
        int(dt_structure_df['Node Count'].sum()),
    'all_structural_checks_passed': True,
    'class_alignment_verified': True,
    'dense_matrix_conversion': False,
    'model_path': str(MODEL_PATH.relative_to(PROJECT_ROOT)),
    'model_size_mib': float(model_size_mib),
    'model_sha256': model_hash,
    'model_reconstruction_verified': False,
    'validation_evaluation_completed': False,
    'threshold_optimization_completed': False,
    'official_test_used': False,
    'status': 'Trained'
}

with open(
    RUN_LOGS_DIR / 'training_log.json',
    'w'
) as file:

    json.dump(
        training_log,
        file,
        indent=4
    )


# Summarize training and structural verification

print("\nDecision Tree training summary")

print("Run ID:", RUN_ID)

print("Binary classifiers trained:",
      len(dt_model.estimators_))

print("Structural checks passed:",
      dt_structure_df['Structural Verification'].eq('Passed').sum(),
      "/", len(LABELS))

print("Maximum observed depth:",
      dt_structure_df['Actual Depth'].max())

print("Total leaves:",
      dt_structure_df['Leaf Count'].sum())

print("Total nodes:",
      dt_structure_df['Node Count'].sum())

print("Classifiers reaching depth limit:",
      dt_structure_df['Depth Limit Reached'].sum())

print("Classifiers reaching leaf limit:",
      dt_structure_df['Leaf Limit Reached'].sum())

print("Minimum observed leaf samples:",
      dt_structure_df['Minimum Leaf Samples'].min())

print("Model fitting time (s):",
      round(model_fit_time, 6))

print("Total training time (s):",
      round(training_time, 6))

print("Training warnings:",
      len(training_warning_records))

print("Model size (MiB):",
      round(model_size_mib, 6))

print("Model saved:", MODEL_PATH)
print("Class alignment: Verified")
print("Dense matrix conversion: No")
print("Model reconstruction: Pending")
print("Validation evaluation: Pending")
print("Threshold optimization: Pending")
print("Official test dataset: Not loaded")

print("\nDecision Tree training completed successfully.")

In [ ]:
# Generate training and validation probability scores

assert MODEL_PATH.is_file()
assert calculate_sha256(MODEL_PATH) == model_hash
assert len(dt_model.estimators_) == len(LABELS)

prediction_start = time.perf_counter()

val_scores = dt_model.predict_proba(X_val)

validation_prediction_time = (
    time.perf_counter() - prediction_start
)

prediction_start = time.perf_counter()

train_scores = dt_model.predict_proba(X_train)

training_prediction_time = (
    time.perf_counter() - prediction_start
)

train_scores = np.asarray(train_scores, dtype=np.float64)
val_scores = np.asarray(val_scores, dtype=np.float64)

for scores, targets in [
    (train_scores, y_train),
    (val_scores, y_val)
]:

    assert scores.shape == targets.shape
    assert np.isfinite(scores).all()
    assert (scores >= 0).all()
    assert (scores <= 1).all()


# Verify positive-class probability alignment

for i, estimator in enumerate(dt_model.estimators_):

    assert np.array_equal(
        estimator.classes_,
        np.array([0, 1])
    )

    individual_scores = estimator.predict_proba(
        X_val[:100]
    )[:, 1]

    np.testing.assert_allclose(
        val_scores[:100, i],
        individual_scores,
        rtol=0,
        atol=1e-12
    )

print("Probability score alignment: Verified")
print("Training score shape:", train_scores.shape)
print("Validation score shape:", val_scores.shape)


# Inspect probability score distributions

score_distribution_records = []

for i, label in enumerate(LABELS):

    scores = val_scores[:, i]

    unique_scores, counts = np.unique(
        scores,
        return_counts=True
    )

    score_distribution_records.append({
        'Label': label,
        'Minimum Score': float(scores.min()),
        'Maximum Score': float(scores.max()),
        'Mean Score': float(scores.mean()),
        'Standard Deviation': float(scores.std()),
        'Median Score': float(np.median(scores)),
        'Unique Scores': len(unique_scores),
        'Exact Zero Scores': int((scores == 0).sum()),
        'Exact One Scores': int((scores == 1).sum()),
        'Exact Default Threshold Scores':
            int((scores == DEFAULT_THRESHOLD).sum()),
        'Most Frequent Score':
            float(unique_scores[np.argmax(counts)]),
        'Most Frequent Score Count':
            int(counts.max()),
        'Most Frequent Score Share (%)':
            float(100 * counts.max() / len(scores))
    })

score_distribution_df = pd.DataFrame(
    score_distribution_records
)

print("\nValidation probability score distribution")

display(score_distribution_df.round(6))

score_distribution_df.to_csv(
    RUN_TABLES_DIR / 'probability_score_distribution.csv',
    index=False
)


# Evaluate both partitions using the default threshold

train_default_metrics, train_default_per_label = (
    evaluate_multilabel(
        y_train,
        train_scores,
        LABELS,
        thresholds=DEFAULT_THRESHOLD
    )
)

default_metrics, default_per_label_df = (
    evaluate_multilabel(
        y_val,
        val_scores,
        LABELS,
        thresholds=DEFAULT_THRESHOLD
    )
)


# Optimize thresholds using validation data only

threshold_df = optimize_thresholds(
    y_val,
    val_scores,
    LABELS
)

assert threshold_df['Label'].tolist() == LABELS

optimized_thresholds = threshold_df[
    'Optimal Threshold'
].to_numpy(dtype=np.float64)

assert optimized_thresholds.shape == (len(LABELS),)
assert np.isfinite(optimized_thresholds).all()
assert (optimized_thresholds >= 0).all()
assert (optimized_thresholds <= 1).all()


# Evaluate validation performance at optimized thresholds

optimized_metrics, optimized_per_label_df = (
    evaluate_multilabel(
        y_val,
        val_scores,
        LABELS,
        thresholds=optimized_thresholds
    )
)

assert np.all(
    optimized_per_label_df['F1'].to_numpy()
    >= default_per_label_df['F1'].to_numpy() - 1e-12
)

np.testing.assert_allclose(
    optimized_per_label_df['F1'].to_numpy(),
    threshold_df['Validation F1'].to_numpy(),
    rtol=0,
    atol=1e-12
)

for metric in [
    'Macro AP',
    'Micro AP',
    'Macro ROC-AUC',
    'Micro ROC-AUC'
]:

    assert np.isclose(
        default_metrics[metric],
        optimized_metrics[metric],
        rtol=0,
        atol=1e-12
    )


# Apply the same validation-selected thresholds to training scores

train_optimized_metrics, train_optimized_per_label = (
    evaluate_multilabel(
        y_train,
        train_scores,
        LABELS,
        thresholds=optimized_thresholds
    )
)


# Compare default and optimized validation performance

overall_comparison_df = pd.DataFrame([
    {
        'Metric': metric,
        'Default': default_metrics[metric],
        'Optimized': optimized_metrics[metric],
        'Change': (
            optimized_metrics[metric]
            - default_metrics[metric]
        )
    }
    for metric in default_metrics
])

print("\nOverall validation performance")

display(overall_comparison_df.round(6))

overall_comparison_df.to_csv(
    RUN_TABLES_DIR / 'overall_threshold_comparison.csv',
    index=False
)


# Compare individual toxicity labels

per_label_comparison_df = pd.DataFrame({
    'Label': LABELS,
    'Average Precision':
        optimized_per_label_df['Average Precision'],
    'ROC-AUC':
        optimized_per_label_df['ROC-AUC'],
    'Default Threshold': DEFAULT_THRESHOLD,
    'Optimized Threshold': optimized_thresholds,
    'Default Precision':
        default_per_label_df['Precision'],
    'Optimized Precision':
        optimized_per_label_df['Precision'],
    'Default Recall':
        default_per_label_df['Recall'],
    'Optimized Recall':
        optimized_per_label_df['Recall'],
    'Default F1':
        default_per_label_df['F1'],
    'Optimized F1':
        optimized_per_label_df['F1'],
    'Actual Positives':
        optimized_per_label_df['Positive Support']
})

per_label_comparison_df['F1 Improvement'] = (
    per_label_comparison_df['Optimized F1']
    - per_label_comparison_df['Default F1']
)

print("\nPer-label validation performance")

display(per_label_comparison_df.round(6))

per_label_comparison_df.to_csv(
    RUN_TABLES_DIR / 'per_label_threshold_comparison.csv',
    index=False
)


# Examine training-validation generalization gaps

generalization_records = []

for threshold_type, train_metrics, val_metrics in [
    (
        'Default',
        train_default_metrics,
        default_metrics
    ),
    (
        'Validation-optimized',
        train_optimized_metrics,
        optimized_metrics
    )
]:

    for metric in [
        'Macro AP',
        'Macro ROC-AUC',
        'Macro Precision',
        'Macro Recall',
        'Macro F1',
        'Micro F1'
    ]:

        generalization_records.append({
            'Threshold Type': threshold_type,
            'Metric': metric,
            'Training': train_metrics[metric],
            'Validation': val_metrics[metric],
            'Training-Validation Gap': (
                train_metrics[metric]
                - val_metrics[metric]
            )
        })

generalization_df = pd.DataFrame(
    generalization_records
)

print("\nTraining-validation generalization comparison")

display(generalization_df.round(6))

generalization_df.to_csv(
    RUN_TABLES_DIR / 'training_validation_comparison.csv',
    index=False
)


# Examine per-label generalization gaps

per_label_generalization_df = pd.DataFrame({
    'Label': LABELS,
    'Training AP':
        train_default_per_label['Average Precision'],
    'Validation AP':
        default_per_label_df['Average Precision'],
    'Training Default F1':
        train_default_per_label['F1'],
    'Validation Default F1':
        default_per_label_df['F1'],
    'Training F1 at Validation Threshold':
        train_optimized_per_label['F1'],
    'Validation Optimized F1':
        optimized_per_label_df['F1']
})

per_label_generalization_df['AP Gap'] = (
    per_label_generalization_df['Training AP']
    - per_label_generalization_df['Validation AP']
)

per_label_generalization_df['Default F1 Gap'] = (
    per_label_generalization_df['Training Default F1']
    - per_label_generalization_df['Validation Default F1']
)

per_label_generalization_df['Optimized F1 Gap'] = (
    per_label_generalization_df[
        'Training F1 at Validation Threshold'
    ]
    - per_label_generalization_df[
        'Validation Optimized F1'
    ]
)

print("\nPer-label generalization gaps")

display(per_label_generalization_df.round(6))

per_label_generalization_df.to_csv(
    RUN_TABLES_DIR / 'per_label_generalization_comparison.csv',
    index=False
)


# Save complete evaluation tables

for filename, metrics in [
    ('default_overall_metrics.csv', default_metrics),
    ('optimized_overall_metrics.csv', optimized_metrics),
    ('training_default_overall_metrics.csv', train_default_metrics),
    ('training_optimized_overall_metrics.csv', train_optimized_metrics)
]:

    pd.DataFrame({
        'Metric': list(metrics.keys()),
        'Value': list(metrics.values())
    }).to_csv(
        RUN_TABLES_DIR / filename,
        index=False
    )

for filename, table in [
    ('default_per_label_metrics.csv', default_per_label_df),
    ('optimized_per_label_metrics.csv', optimized_per_label_df),
    ('training_default_per_label_metrics.csv', train_default_per_label),
    ('training_optimized_per_label_metrics.csv', train_optimized_per_label),
    ('optimized_thresholds.csv', threshold_df),
    ('feature_verification.csv', feature_verification_df)
]:

    table.to_csv(
        RUN_TABLES_DIR / filename,
        index=False
    )


# Save validation scores and binary predictions

default_predictions = apply_thresholds(
    val_scores,
    DEFAULT_THRESHOLD
)

optimized_predictions = apply_thresholds(
    val_scores,
    optimized_thresholds
)

predictions_df = pd.DataFrame({
    'id': val_df['id'].to_numpy()
})

for i, label in enumerate(LABELS):

    predictions_df[f'{label}_true'] = y_val[:, i]

    predictions_df[f'{label}_score'] = val_scores[:, i]

    predictions_df[f'{label}_pred_default'] = (
        default_predictions[:, i].astype(np.int8)
    )

    predictions_df[f'{label}_pred_optimized'] = (
        optimized_predictions[:, i].astype(np.int8)
    )

assert predictions_df.shape == (
    len(y_val),
    1 + 4 * len(LABELS)
)

PREDICTIONS_PATH = (
    RUN_PREDICTIONS_DIR / 'validation_predictions.parquet'
)

predictions_df.to_parquet(
    PREDICTIONS_PATH,
    index=False
)

saved_predictions_df = pd.read_parquet(
    PREDICTIONS_PATH
)

pd.testing.assert_frame_equal(
    predictions_df,
    saved_predictions_df,
    check_exact=True
)


# Save full-precision validation thresholds

threshold_config = {
    'run_id': RUN_ID,
    'labels': LABELS,
    'score_type': SCORE_TYPE,
    'default_threshold': DEFAULT_THRESHOLD,
    'optimized_thresholds': {
        label: float(threshold)
        for label, threshold in zip(
            LABELS,
            optimized_thresholds
        )
    },
    'selection_partition': 'validation',
    'optimization_metric': 'per_label_f1',
    'tie_breaking': 'highest_threshold',
    'comparison_operator': '>=',
    'probability_calibration': False,
    'official_test_used': False
}

THRESHOLD_PATH = (
    RUN_CONFIGS_DIR / f'{RUN_ID}_thresholds.json'
)

with open(THRESHOLD_PATH, 'w') as file:

    json.dump(
        threshold_config,
        file,
        indent=4
    )

with open(THRESHOLD_PATH, 'r') as file:
    saved_threshold_config = json.load(file)

assert saved_threshold_config == threshold_config

reloaded_thresholds = np.array([
    saved_threshold_config['optimized_thresholds'][label]
    for label in LABELS
], dtype=np.float64)

assert np.array_equal(
    optimized_thresholds,
    reloaded_thresholds
)


# Reload the saved model and reconstruct validation scores

assert calculate_sha256(MODEL_PATH) == model_hash

reloaded_dt_model = joblib.load(MODEL_PATH)

reconstructed_scores = np.asarray(
    reloaded_dt_model.predict_proba(X_val),
    dtype=np.float64
)

np.testing.assert_allclose(
    val_scores,
    reconstructed_scores,
    rtol=0,
    atol=1e-12
)

maximum_reconstruction_error = float(
    np.abs(
        val_scores - reconstructed_scores
    ).max()
)

reconstructed_default_predictions = apply_thresholds(
    reconstructed_scores,
    DEFAULT_THRESHOLD
)

reconstructed_optimized_predictions = apply_thresholds(
    reconstructed_scores,
    reloaded_thresholds
)

assert np.array_equal(
    default_predictions,
    reconstructed_default_predictions
)

assert np.array_equal(
    optimized_predictions,
    reconstructed_optimized_predictions
)


# Verify metrics reconstructed from the saved predictions

saved_targets = saved_predictions_df[[
    f'{label}_true'
    for label in LABELS
]].to_numpy()

saved_scores = saved_predictions_df[[
    f'{label}_score'
    for label in LABELS
]].to_numpy(dtype=np.float64)

assert saved_predictions_df['id'].tolist() == val_df['id'].tolist()

assert np.array_equal(saved_targets, y_val)
assert np.array_equal(saved_scores, val_scores)

for threshold_values, original_metrics, original_per_label in [
    (
        DEFAULT_THRESHOLD,
        default_metrics,
        default_per_label_df
    ),
    (
        reloaded_thresholds,
        optimized_metrics,
        optimized_per_label_df
    )
]:

    reconstructed_metrics, reconstructed_per_label = (
        evaluate_multilabel(
            saved_targets,
            saved_scores,
            LABELS,
            thresholds=threshold_values
        )
    )

    for metric in original_metrics:

        assert np.isclose(
            original_metrics[metric],
            reconstructed_metrics[metric],
            rtol=0,
            atol=1e-12
        )

    pd.testing.assert_frame_equal(
        original_per_label,
        reconstructed_per_label,
        check_exact=False,
        rtol=1e-12,
        atol=1e-12
    )


# Save the completed run configuration

run_config = {
    **dt_experiment_config,
    'feature_count': FEATURE_BUDGET,
    'model_path': str(
        MODEL_PATH.relative_to(PROJECT_ROOT)
    ),
    'threshold_path': str(
        THRESHOLD_PATH.relative_to(PROJECT_ROOT)
    ),
    'predictions_path': str(
        PREDICTIONS_PATH.relative_to(PROJECT_ROOT)
    )
}

RUN_CONFIG_PATH = RUN_CONFIGS_DIR / f'{RUN_ID}.json'

with open(RUN_CONFIG_PATH, 'w') as file:

    json.dump(
        run_config,
        file,
        indent=4
    )

with open(RUN_CONFIG_PATH, 'r') as file:
    assert json.load(file) == run_config


# Construct the experiment summary

run_summary = {
    'Run ID': RUN_ID,
    'Model': MODEL_NAME,
    'Feature Representation': REPRESENTATION,
    'Class Weight': 'None',
    'Training Samples': len(y_train),
    'Validation Samples': len(y_val),
    'Features': FEATURE_BUDGET,
    'Macro AP': optimized_metrics['Macro AP'],
    'Micro AP': optimized_metrics['Micro AP'],
    'Macro ROC-AUC': optimized_metrics['Macro ROC-AUC'],
    'Default Macro F1': default_metrics['Macro F1'],
    'Optimized Macro F1': optimized_metrics['Macro F1'],
    'Default Micro F1': default_metrics['Micro F1'],
    'Optimized Micro F1': optimized_metrics['Micro F1'],
    'Optimized Macro Precision': optimized_metrics['Macro Precision'],
    'Optimized Macro Recall': optimized_metrics['Macro Recall'],
    'Training Macro AP': train_default_metrics['Macro AP'],
    'Training Default Macro F1': train_default_metrics['Macro F1'],
    'Macro AP Generalization Gap': (
        train_default_metrics['Macro AP']
        - default_metrics['Macro AP']
    ),
    'Training Time (s)': training_time,
    'Model Fit Time (s)': model_fit_time,
    'Validation Prediction Time (s)': validation_prediction_time,
    'Training Prediction Time (s)': training_prediction_time,
    'Feature Loading Time (s)': feature_loading_time,
    'Model Size (MiB)': model_size_mib,
    'Maximum Observed Depth': int(dt_structure_df['Actual Depth'].max()),
    'Total Leaves': int(dt_structure_df['Leaf Count'].sum()),
    'Total Nodes': int(dt_structure_df['Node Count'].sum()),
    'Training Warnings': len(training_warning_records),
    'Maximum Score Reconstruction Error': maximum_reconstruction_error,
    'Structural Verification': 'Passed',
    'Official Test Used': False,
    'Status': 'Completed'
}

run_summary_df = pd.DataFrame([run_summary])

run_summary_df.to_csv(
    RUN_TABLES_DIR / 'run_summary.csv',
    index=False
)

print("\nDecision Tree experiment summary")

display(run_summary_df.T)


# Update training and evaluation metadata

evaluation_end_time = datetime.now(
    timezone.utc
).isoformat()

training_log.update({
    'validation_prediction_time_seconds':
        float(validation_prediction_time),
    'training_prediction_time_seconds':
        float(training_prediction_time),
    'evaluation_end_time': evaluation_end_time,
    'maximum_score_reconstruction_error':
        maximum_reconstruction_error,
    'model_reconstruction_verified': True,
    'metric_reconstruction_verified': True,
    'score_alignment_verified': True,
    'prediction_roundtrip_verified': True,
    'threshold_roundtrip_verified': True,
    'validation_evaluation_completed': True,
    'threshold_optimization_completed': True,
    'training_evaluation_completed': True,
    'official_test_used': False,
    'status': 'Completed'
})

TRAINING_LOG_PATH = (
    RUN_LOGS_DIR / 'training_log.json'
)

with open(TRAINING_LOG_PATH, 'w') as file:

    json.dump(
        training_log,
        file,
        indent=4
    )


# Fingerprint critical experiment artifacts

critical_artifacts = [
    ('Model', MODEL_PATH),
    ('Run Configuration', RUN_CONFIG_PATH),
    ('Threshold Configuration', THRESHOLD_PATH),
    ('Validation Predictions', PREDICTIONS_PATH),
    ('Training Log', TRAINING_LOG_PATH),
    ('Feature Manifest', FEATURES_DIR / 'feature_artifact_manifest.csv'),
    ('Evaluation Module', SRC_DIR / 'evaluation_utils.py')
]

artifact_records = []

for artifact_type, path in critical_artifacts:

    assert path.is_file()
    assert path.stat().st_size > 0

    artifact_records.append({
        'Run ID': RUN_ID,
        'Artifact Type': artifact_type,
        'Path': str(path.relative_to(PROJECT_ROOT)),
        'Size Bytes': path.stat().st_size,
        'SHA256': calculate_sha256(path)
    })

artifact_manifest_df = pd.DataFrame(
    artifact_records
)

artifact_manifest_df.to_csv(
    RUN_TABLES_DIR / 'artifact_manifest.csv',
    index=False
)

assert calculate_sha256(
    SRC_DIR / 'evaluation_utils.py'
) == evaluation_hash

assert calculate_sha256(
    FEATURES_DIR / 'feature_artifact_manifest.csv'
) == feature_manifest_hash


# Store the completed experiment for subsequent analysis

dt_results[RUN_ID] = {
    'spec': experiment_spec.copy(),
    'summary': run_summary,
    'default_metrics': default_metrics,
    'optimized_metrics': optimized_metrics,
    'default_per_label': default_per_label_df,
    'optimized_per_label': optimized_per_label_df,
    'thresholds': threshold_df,
    'model_path': MODEL_PATH,
    'config_path': RUN_CONFIG_PATH,
    'threshold_path': THRESHOLD_PATH,
    'predictions_path': PREDICTIONS_PATH,
    'results_path': RUN_DIR
}

del reloaded_dt_model
del reconstructed_scores
del saved_predictions_df

gc.collect()


# Summarize evaluation and generalization performance

print("\nDecision Tree evaluation summary")

print("Training Macro AP:",
      round(train_default_metrics['Macro AP'], 6))

print("Validation Macro AP:",
      round(default_metrics['Macro AP'], 6))

print("Macro AP generalization gap:",
      round(run_summary['Macro AP Generalization Gap'], 6))

print("Training default Macro F1:",
      round(train_default_metrics['Macro F1'], 6))

print("Validation default Macro F1:",
      round(default_metrics['Macro F1'], 6))

print("Validation optimized Macro F1:",
      round(optimized_metrics['Macro F1'], 6))

print("Validation optimized Micro F1:",
      round(optimized_metrics['Micro F1'], 6))

print("Validation Macro ROC-AUC:",
      round(optimized_metrics['Macro ROC-AUC'], 6))

print("Macro F1 improvement:",
      round(
          optimized_metrics['Macro F1']
          - default_metrics['Macro F1'],
          6
      ))

print("Labels with improved F1:",
      int((per_label_comparison_df['F1 Improvement'] > 0).sum()),
      "/", len(LABELS))

print("Validation prediction time (s):",
      round(validation_prediction_time, 6))

print("Maximum score reconstruction error:",
      maximum_reconstruction_error)

print("Probability score alignment: Passed")
print("Model reconstruction: Passed")
print("Metric reconstruction: Passed")
print("Threshold roundtrip: Passed")
print("Prediction roundtrip: Passed")
print("Critical artifacts fingerprinted:", len(artifact_manifest_df))
print("Completed Decision Tree experiments:", len(dt_results))
print("Experiment registration: Pending")
print("Official test dataset: Not loaded")

print("\nDecision Tree evaluation completed successfully.")

In [ ]:
# Define the common word-feature baseline experiments

baseline_run_ids = [
    'NB_WORD_BASELINE_S42',
    'LR_WORD_BASELINE_S42',
    'SVM_WORD_BASELINE_S42',
    RUN_ID
]

baseline_results = {}
baseline_summary_records = []
baseline_per_label_records = []

for run_id in baseline_run_ids:

    if run_id == RUN_ID:

        current_config = run_config

        model_name = MODEL_NAME
        model_path = MODEL_PATH
        predictions_path = PREDICTIONS_PATH
        fit_time = training_time

        expected_ap = run_summary['Macro AP']
        expected_f1 = run_summary['Optimized Macro F1']

    else:

        record = registry_df.loc[
            registry_df['run_id'] == run_id
        ].iloc[0]

        current_config = previous_run_configs[run_id]

        model_name = record['model']

        model_path = PROJECT_ROOT / record['model_path']

        previous_run_dir = (
            PROJECT_ROOT / record['results_path']
        )

        predictions_dir = (
            previous_run_dir / 'predictions'
        )

        candidate_paths = [
            predictions_dir / 'validation_predictions.parquet',
            predictions_dir / f'{run_id}_validation_predictions.parquet'
        ]

        available_paths = [
            path for path in candidate_paths
            if path.is_file()
        ]

        assert len(available_paths) == 1, (
            f'Prediction file resolution failed: {run_id}'
        )

        predictions_path = available_paths[0]

        fit_time = float(record['training_time_seconds'])

        expected_ap = float(record['validation_macro_ap'])
        expected_f1 = float(record['validation_macro_f1'])


    # Verify identical experimental conditions

    assert current_config['feature_representation'] == REPRESENTATION

    assert current_config['training_dataset_sha256'] == (
        EXPECTED_TRAINING_HASH
    )

    assert current_config['validation_dataset_sha256'] == (
        EXPECTED_VALIDATION_HASH
    )

    assert current_config['evaluation_module_sha256'] == (
        evaluation_hash
    )

    assert current_config['feature_manifest_sha256'] == (
        feature_manifest_hash
    )

    assert current_config['labels'] == LABELS
    assert current_config['seed'] == SEED
    assert not current_config['official_test_used']

    assert model_path.is_file()
    assert predictions_path.is_file()


    # Reload validation targets and scores

    baseline_predictions = pd.read_parquet(
        predictions_path
    )

    assert (
        baseline_predictions['id'].tolist()
        == val_df['id'].tolist()
    )

    targets = baseline_predictions[[
        f'{label}_true'
        for label in LABELS
    ]].to_numpy()

    scores = baseline_predictions[[
        f'{label}_score'
        for label in LABELS
    ]].to_numpy(dtype=np.float64)

    assert np.array_equal(targets, y_val)
    assert scores.shape == y_val.shape
    assert np.isfinite(scores).all()

    default_threshold = (
        0.0 if model_name == 'Linear SVM'
        else 0.5
    )

    if model_name != 'Linear SVM':

        assert (scores >= 0).all()
        assert (scores <= 1).all()


    # Reconstruct default and optimized validation metrics

    baseline_default_metrics, baseline_default_labels = (
        evaluate_multilabel(
            targets,
            scores,
            LABELS,
            thresholds=default_threshold
        )
    )

    baseline_threshold_df = optimize_thresholds(
        targets,
        scores,
        LABELS
    )

    baseline_thresholds = baseline_threshold_df[
        'Optimal Threshold'
    ].to_numpy(dtype=np.float64)

    baseline_optimized_metrics, baseline_optimized_labels = (
        evaluate_multilabel(
            targets,
            scores,
            LABELS,
            thresholds=baseline_thresholds
        )
    )


    # Verify reconstructed binary predictions

    saved_default = baseline_predictions[[
        f'{label}_pred_default'
        for label in LABELS
    ]].to_numpy()

    saved_optimized = baseline_predictions[[
        f'{label}_pred_optimized'
        for label in LABELS
    ]].to_numpy()

    assert np.array_equal(
        saved_default,
        apply_thresholds(scores, default_threshold)
    )

    assert np.array_equal(
        saved_optimized,
        apply_thresholds(scores, baseline_thresholds)
    )

    assert np.isclose(
        baseline_optimized_metrics['Macro AP'],
        expected_ap,
        rtol=0,
        atol=1e-6
    )

    assert np.isclose(
        baseline_optimized_metrics['Macro F1'],
        expected_f1,
        rtol=0,
        atol=1e-6
    )


    # Record reconstructed model performance

    baseline_summary_records.append({
        'Run ID': run_id,
        'Model': model_name,
        'Representation': REPRESENTATION,
        'Training Samples': len(y_train),
        'Validation Samples': len(y_val),
        'Features': FEATURE_BUDGET,
        'Macro AP': baseline_optimized_metrics['Macro AP'],
        'Macro ROC-AUC':
            baseline_optimized_metrics['Macro ROC-AUC'],
        'Default Macro F1':
            baseline_default_metrics['Macro F1'],
        'Optimized Macro F1':
            baseline_optimized_metrics['Macro F1'],
        'Default Micro F1':
            baseline_default_metrics['Micro F1'],
        'Optimized Micro F1':
            baseline_optimized_metrics['Micro F1'],
        'Optimized Macro Precision':
            baseline_optimized_metrics['Macro Precision'],
        'Optimized Macro Recall':
            baseline_optimized_metrics['Macro Recall'],
        'Optimized Hamming Loss':
            baseline_optimized_metrics['Hamming Loss'],
        'Optimized Exact Match Accuracy':
            baseline_optimized_metrics['Exact Match Accuracy'],
        'Training Time (s)': fit_time,
        'Model Size (MiB)':
            model_path.stat().st_size / (1024 ** 2),
        'Metric Reconstruction': 'Passed',
        'Official Test Used': False
    })

    label_metrics = baseline_optimized_labels.copy()

    label_metrics.insert(0, 'Model', model_name)
    label_metrics.insert(0, 'Run ID', run_id)

    label_metrics['Default F1'] = (
        baseline_default_labels['F1'].to_numpy()
    )

    baseline_per_label_records.append(
        label_metrics
    )

    baseline_results[run_id] = {
        'model': model_name,
        'default_metrics': baseline_default_metrics,
        'optimized_metrics': baseline_optimized_metrics,
        'default_per_label': baseline_default_labels,
        'optimized_per_label': baseline_optimized_labels
    }

    print(f'{run_id}: Reconstruction passed')


# Consolidate the four common-feature baselines

dt_baseline_comparison_df = pd.DataFrame(
    baseline_summary_records
)

assert len(dt_baseline_comparison_df) == 4

assert dt_baseline_comparison_df[
    'Metric Reconstruction'
].eq('Passed').all()

dt_baseline_comparison_df.to_csv(
    TABLES_DIR / 'dt_common_baseline_comparison.csv',
    index=False
)

print("\nCommon Word TF-IDF baseline comparison")

display(
    dt_baseline_comparison_df[[
        'Model',
        'Macro AP',
        'Macro ROC-AUC',
        'Default Macro F1',
        'Optimized Macro F1',
        'Optimized Micro F1',
        'Training Time (s)',
        'Model Size (MiB)'
    ]].round(6)
)


# Consolidate per-label results

dt_baseline_per_label_df = pd.concat(
    baseline_per_label_records,
    ignore_index=True
)

assert len(dt_baseline_per_label_df) == 4 * len(LABELS)

dt_baseline_per_label_df.to_csv(
    TABLES_DIR / 'dt_common_baseline_per_label.csv',
    index=False
)

model_order = (
    dt_baseline_comparison_df['Model'].tolist()
)

ap_comparison_df = (
    dt_baseline_per_label_df.pivot(
        index='Label',
        columns='Model',
        values='Average Precision'
    )
    .reindex(
        index=LABELS,
        columns=model_order
    )
)

f1_comparison_df = (
    dt_baseline_per_label_df.pivot(
        index='Label',
        columns='Model',
        values='F1'
    )
    .reindex(
        index=LABELS,
        columns=model_order
    )
)

assert not ap_comparison_df.isna().any().any()
assert not f1_comparison_df.isna().any().any()

print("\nPer-label Average Precision comparison")

display(ap_comparison_df.round(6))

print("\nPer-label optimized F1 comparison")

display(f1_comparison_df.round(6))

ap_comparison_df.to_csv(
    TABLES_DIR / 'dt_common_baseline_ap_comparison.csv'
)

f1_comparison_df.to_csv(
    TABLES_DIR / 'dt_common_baseline_f1_comparison.csv'
)


# Rank common-feature baselines

dt_baseline_ranking_df = (
    dt_baseline_comparison_df[[
        'Run ID',
        'Model',
        'Macro AP',
        'Macro ROC-AUC',
        'Optimized Macro F1',
        'Optimized Micro F1',
        'Training Time (s)',
        'Model Size (MiB)'
    ]]
    .sort_values(
        ['Macro AP', 'Optimized Macro F1', 'Run ID'],
        ascending=[False, False, True]
    )
    .reset_index(drop=True)
)

dt_baseline_ranking_df.insert(
    0,
    'Rank',
    np.arange(1, len(dt_baseline_ranking_df) + 1)
)

dt_baseline_ranking_df.to_csv(
    TABLES_DIR / 'dt_common_baseline_ranking.csv',
    index=False
)

print("\nCommon-feature baseline ranking")

display(dt_baseline_ranking_df.round(6))


# Prepare consistent visualization labels

short_model_names = {
    'Multinomial Naive Bayes': 'Naive Bayes',
    'Logistic Regression': 'Logistic Regression',
    'Linear SVM': 'Linear SVM',
    'Decision Tree': 'Decision Tree'
}

label_names = [
    label.replace('_', ' ').title()
    for label in LABELS
]

plot_colors = [
    MODEL_COLORS[model]
    for model in model_order
]

plot_df = dt_baseline_comparison_df.set_index(
    'Model'
).loc[model_order].copy()

plot_df.index = [
    short_model_names[model]
    for model in plot_df.index
]


# Figure 1: Overall common-feature model comparison

fig, axes = plt.subplots(
    2, 2,
    figsize=(15, 10)
)

for ax, metric in zip(
    axes.flat,
    [
        'Macro AP',
        'Macro ROC-AUC',
        'Optimized Macro F1',
        'Optimized Micro F1'
    ]
):

    bars = ax.bar(
        plot_df.index,
        plot_df[metric],
        color=plot_colors,
        width=0.65
    )

    ax.bar_label(
        bars,
        fmt='%.4f',
        padding=4,
        fontsize=10
    )

    ax.set_title(metric)
    ax.set_ylabel('Score')
    ax.set_ylim(0, 1.1)
    ax.tick_params(axis='x', rotation=15)
    ax.grid(axis='x', visible=False)

fig.suptitle(
    'Classical ML Comparison Using Common Word TF-IDF Features',
    fontsize=16
)

sns.despine()

fig.tight_layout(
    rect=[0, 0, 1, 0.95]
)

fig.savefig(
    FIGURES_DIR / 'dt_common_baseline_overall_comparison.png',
    dpi=300,
    bbox_inches='tight'
)

plt.show()
plt.close(fig)


# Figure 2: Per-label baseline comparison

fig, axes = plt.subplots(
    2, 1,
    figsize=(16, 11)
)

for ax, table, title in [
    (
        axes[0],
        ap_comparison_df,
        'Per-Label Average Precision'
    ),
    (
        axes[1],
        f1_comparison_df,
        'Per-Label Optimized F1'
    )
]:

    chart_df = table.copy()

    chart_df.index = label_names

    chart_df.columns = [
        short_model_names[model]
        for model in chart_df.columns
    ]

    chart_df.plot(
        kind='bar',
        ax=ax,
        color=plot_colors,
        width=0.8
    )

    for container in ax.containers:

        ax.bar_label(
            container,
            fmt='%.3f',
            padding=3,
            fontsize=8
        )

    ax.set_title(title)
    ax.set_ylabel('Score')
    ax.set_xlabel('')
    ax.set_ylim(0, 1.1)
    ax.tick_params(axis='x', rotation=0)
    ax.grid(axis='x', visible=False)
    ax.legend(fontsize=9)

fig.suptitle(
    'Classical ML: Per-Label Word TF-IDF Baseline Comparison',
    fontsize=16
)

sns.despine()

fig.tight_layout(
    rect=[0, 0, 1, 0.95]
)

fig.savefig(
    FIGURES_DIR / 'dt_common_baseline_per_label_comparison.png',
    dpi=300,
    bbox_inches='tight'
)

plt.show()
plt.close(fig)


# Combine Decision Tree generalization and score diagnostics

diagnostic_df = (
    per_label_generalization_df.merge(
        score_distribution_df[[
            'Label',
            'Unique Scores',
            'Most Frequent Score',
            'Most Frequent Score Count',
            'Most Frequent Score Share (%)'
        ]],
        on='Label',
        validate='one_to_one'
    )
    .merge(
        dt_structure_df[[
            'Label',
            'Actual Depth',
            'Leaf Count',
            'Node Count',
            'Maximum Leaf Samples',
            'Median Leaf Samples',
            'Unique Split Features'
        ]],
        on='Label',
        validate='one_to_one'
    )
)

assert len(diagnostic_df) == len(LABELS)
assert diagnostic_df['Label'].tolist() == LABELS

diagnostic_df.to_csv(
    TABLES_DIR / 'dt_generalization_and_score_diagnostics.csv',
    index=False
)

print("\nDecision Tree generalization and score diagnostics")

display(diagnostic_df.round(6))


# Figure 3: Generalization and probability concentration

fig, axes = plt.subplots(
    2, 2,
    figsize=(16, 11)
)

diagnostic_plot_df = diagnostic_df.set_index(
    'Label'
).loc[LABELS].copy()

diagnostic_plot_df.index = label_names

for ax, columns, title in [
    (
        axes[0, 0],
        ['Training AP', 'Validation AP'],
        'Training vs Validation Average Precision'
    ),
    (
        axes[0, 1],
        ['Training Default F1', 'Validation Default F1'],
        'Training vs Validation Default F1'
    )
]:

    diagnostic_plot_df[columns].plot(
        kind='bar',
        ax=ax,
        color=['#4C78A8', '#F58518'],
        width=0.7
    )

    for container in ax.containers:

        ax.bar_label(
            container,
            fmt='%.3f',
            padding=3,
            fontsize=8
        )

    ax.set_title(title)
    ax.set_xlabel('')
    ax.set_ylabel('Score')
    ax.set_ylim(0, 1.1)
    ax.tick_params(axis='x', rotation=15)
    ax.grid(axis='x', visible=False)
    ax.legend(fontsize=9)


# Visualize the number of distinct validation scores

bars = axes[1, 0].bar(
    label_names,
    diagnostic_df['Unique Scores'],
    color=[
        LABEL_COLORS[label]
        for label in LABELS
    ]
)

axes[1, 0].bar_label(
    bars,
    padding=3,
    fontsize=10
)

axes[1, 0].set_title(
    'Number of Unique Validation Probability Scores'
)

axes[1, 0].set_ylabel('Unique scores')

axes[1, 0].set_ylim(
    0,
    diagnostic_df['Unique Scores'].max() * 1.2
)

axes[1, 0].tick_params(axis='x', rotation=15)
axes[1, 0].grid(axis='x', visible=False)


# Visualize concentration around the most frequent score

bars = axes[1, 1].bar(
    label_names,
    diagnostic_df['Most Frequent Score Share (%)'],
    color=[
        LABEL_COLORS[label]
        for label in LABELS
    ]
)

axes[1, 1].bar_label(
    bars,
    fmt='%.1f%%',
    padding=3,
    fontsize=9
)

axes[1, 1].set_title(
    'Validation Probability Score Concentration'
)

axes[1, 1].set_ylabel(
    'Comments receiving most frequent score (%)'
)

axes[1, 1].set_ylim(0, 112)
axes[1, 1].tick_params(axis='x', rotation=15)
axes[1, 1].grid(axis='x', visible=False)

fig.suptitle(
    'Decision Tree: Generalization and Probability Resolution',
    fontsize=16
)

sns.despine()

fig.tight_layout(
    rect=[0, 0, 1, 0.95]
)

fig.savefig(
    FIGURES_DIR / 'dt_generalization_and_score_diagnostics.png',
    dpi=300,
    bbox_inches='tight'
)

plt.show()
plt.close(fig)


# Record Decision Tree findings against previous baselines

dt_rank = int(
    dt_baseline_ranking_df.loc[
        dt_baseline_ranking_df['Run ID'] == RUN_ID,
        'Rank'
    ].iloc[0]
)

nb_ap = baseline_results[
    'NB_WORD_BASELINE_S42'
]['optimized_metrics']['Macro AP']

lr_ap = baseline_results[
    'LR_WORD_BASELINE_S42'
]['optimized_metrics']['Macro AP']

svm_ap = baseline_results[
    'SVM_WORD_BASELINE_S42'
]['optimized_metrics']['Macro AP']

largest_gap_row = diagnostic_df.loc[
    diagnostic_df['AP Gap'].idxmax()
]

highest_concentration_row = diagnostic_df.loc[
    diagnostic_df['Most Frequent Score Share (%)'].idxmax()
]

dt_analysis_config = {
    'run_id': RUN_ID,
    'comparison_scope': 'common_word_tfidf_baselines',
    'models_compared': len(baseline_results),
    'primary_selection_metric': 'macro_average_precision',
    'common_baseline_rank': dt_rank,
    'validation_macro_ap': float(run_summary['Macro AP']),
    'training_macro_ap': float(train_default_metrics['Macro AP']),
    'macro_ap_generalization_gap':
        float(run_summary['Macro AP Generalization Gap']),
    'macro_ap_difference_vs_nb':
        float(run_summary['Macro AP'] - nb_ap),
    'macro_ap_difference_vs_lr':
        float(run_summary['Macro AP'] - lr_ap),
    'macro_ap_difference_vs_svm':
        float(run_summary['Macro AP'] - svm_ap),
    'largest_ap_generalization_gap_label':
        str(largest_gap_row['Label']),
    'largest_ap_generalization_gap':
        float(largest_gap_row['AP Gap']),
    'highest_score_concentration_label':
        str(highest_concentration_row['Label']),
    'highest_score_concentration_percentage':
        float(
            highest_concentration_row[
                'Most Frequent Score Share (%)'
            ]
        ),
    'all_trees_reached_depth_limit':
        bool(dt_structure_df['Depth Limit Reached'].all()),
    'additional_hyperparameter_tuning_planned': False,
    'retained_as_nonlinear_baseline': True,
    'statistical_significance_established': False,
    'final_project_model_frozen': False,
    'official_test_used': False
}

DT_ANALYSIS_CONFIG_PATH = (
    RUN_CONFIGS_DIR / 'dt_baseline_analysis.json'
)

with open(
    DT_ANALYSIS_CONFIG_PATH, 'w'
) as file:

    json.dump(
        dt_analysis_config,
        file,
        indent=4
    )

with open(
    DT_ANALYSIS_CONFIG_PATH, 'r'
) as file:

    assert json.load(file) == dt_analysis_config


# Summarize the completed analysis

print("\nDecision Tree comparative analysis summary")

print("Common-feature models compared:",
      len(baseline_results))

print("Metric and prediction reconstruction: Passed")

print("Decision Tree common-baseline rank:",
      dt_rank, "/", len(baseline_results))

print("Validation Macro AP:",
      round(run_summary['Macro AP'], 6))

print("Optimized Macro F1:",
      round(run_summary['Optimized Macro F1'], 6))

print("Macro AP difference vs Naive Bayes:",
      round(run_summary['Macro AP'] - nb_ap, 6))

print("Macro AP difference vs Logistic Regression:",
      round(run_summary['Macro AP'] - lr_ap, 6))

print("Macro AP difference vs Linear SVM:",
      round(run_summary['Macro AP'] - svm_ap, 6))

print("Macro AP generalization gap:",
      round(run_summary['Macro AP Generalization Gap'], 6))

print("Largest AP generalization gap label:",
      largest_gap_row['Label'])

print("Highest probability concentration label:",
      highest_concentration_row['Label'])

print("Highest probability concentration (%):",
      round(
          highest_concentration_row[
              'Most Frequent Score Share (%)'
          ],
          6
      ))

print("Research figures generated: 3")
print("Analysis configuration: Saved and verified")
print("Additional model training: No")
print("Experiment registration: Pending")
print("Official test dataset: Not loaded")

gc.collect()

print("\nDecision Tree comparative analysis completed successfully.")

In [ ]:
# Verify completed experiment status

assert len(dt_results) == 1
assert RUN_ID in dt_results

assert run_summary['Status'] == 'Completed'
assert run_summary['Structural Verification'] == 'Passed'
assert not run_summary['Official Test Used']
assert maximum_reconstruction_error <= 1e-12

with open(TRAINING_LOG_PATH, 'r') as file:
    saved_training_log = json.load(file)

for field in [
    'all_structural_checks_passed',
    'class_alignment_verified',
    'model_reconstruction_verified',
    'metric_reconstruction_verified',
    'score_alignment_verified',
    'prediction_roundtrip_verified',
    'threshold_roundtrip_verified',
    'validation_evaluation_completed',
    'threshold_optimization_completed'
]:
    assert saved_training_log[field]

assert saved_training_log['status'] == 'Completed'
assert not saved_training_log['official_test_used']


# Verify critical artifact fingerprints

saved_manifest_df = pd.read_csv(
    RUN_TABLES_DIR / 'artifact_manifest.csv'
)

assert len(saved_manifest_df) == 7
assert saved_manifest_df['Run ID'].eq(RUN_ID).all()
assert saved_manifest_df['Artifact Type'].is_unique

verified_fingerprint_count = 0

for _, artifact in saved_manifest_df.iterrows():

    path = PROJECT_ROOT / artifact['Path']

    assert path.is_file()
    assert path.stat().st_size == artifact['Size Bytes']

    assert calculate_sha256(path) == artifact['SHA256'], (
        f"Artifact fingerprint mismatch: {path}"
    )

    verified_fingerprint_count += 1

print("Critical artifact fingerprints verified:",
      verified_fingerprint_count)


# Verify the six fitted tree structures

saved_structure_df = pd.read_csv(
    RUN_TABLES_DIR / 'classifier_structure_summary.csv'
)

assert saved_structure_df['Label'].tolist() == LABELS

assert saved_structure_df[
    'Structural Verification'
].eq('Passed').all()

assert saved_structure_df[
    'Class Alignment'
].eq('Passed').all()

assert saved_structure_df[
    'Actual Depth'
].le(DT_PARAMS['max_depth']).all()

assert saved_structure_df[
    'Leaf Count'
].le(DT_PARAMS['max_leaf_nodes']).all()

assert saved_structure_df[
    'Minimum Leaf Samples'
].ge(DT_PARAMS['min_samples_leaf']).all()

assert (
    saved_structure_df['Node Count']
    == 2 * saved_structure_df['Leaf Count'] - 1
).all()


# Define required notebook-level artifacts

required_tables = [
    'previous_run_consistency.csv',
    'probability_score_evaluation_verification.csv',
    'environment_versions.csv',
    'environment_compatibility.csv',
    'dataset_verification.csv',
    'feature_representation_verification.csv',
    'previous_feature_consistency.csv',
    'label_support_and_imbalance.csv',
    'dt_hyperparameter_rationale.csv',
    'dt_structural_limits.csv',
    'dt_experiment_plan.csv',
    'dt_common_baseline_comparison.csv',
    'dt_common_baseline_per_label.csv',
    'dt_common_baseline_ap_comparison.csv',
    'dt_common_baseline_f1_comparison.csv',
    'dt_common_baseline_ranking.csv',
    'dt_generalization_and_score_diagnostics.csv'
]

required_run_tables = [
    'classifier_structure_summary.csv',
    'model_complexity_summary.csv',
    'probability_score_distribution.csv',
    'overall_threshold_comparison.csv',
    'per_label_threshold_comparison.csv',
    'training_validation_comparison.csv',
    'per_label_generalization_comparison.csv',
    'default_overall_metrics.csv',
    'optimized_overall_metrics.csv',
    'training_default_overall_metrics.csv',
    'training_optimized_overall_metrics.csv',
    'default_per_label_metrics.csv',
    'optimized_per_label_metrics.csv',
    'training_default_per_label_metrics.csv',
    'training_optimized_per_label_metrics.csv',
    'optimized_thresholds.csv',
    'feature_verification.csv',
    'run_summary.csv',
    'artifact_manifest.csv'
]

required_figures = [
    'dt_common_baseline_overall_comparison.png',
    'dt_common_baseline_per_label_comparison.png',
    'dt_generalization_and_score_diagnostics.png'
]

required_configs = [
    'dt_experiment_config.json',
    'dt_baseline_analysis.json',
    f'{RUN_ID}.json',
    f'{RUN_ID}_thresholds.json'
]


# Construct the artifact verification inventory

verification_records = []

def verify_artifact(path, category):

    available = (
        path.is_file()
        and path.stat().st_size > 0
    )

    verification_records.append({
        'Type': category,
        'Filename': path.name,
        'Path': str(path.relative_to(PROJECT_ROOT)),
        'Size Bytes': (
            path.stat().st_size if available else 0
        ),
        'Status': 'Available' if available else 'Missing'
    })


for filename in required_tables:

    verify_artifact(
        TABLES_DIR / filename,
        'Analysis Table'
    )

for filename in required_run_tables:

    verify_artifact(
        RUN_TABLES_DIR / filename,
        'Run Table'
    )

for filename in required_figures:

    verify_artifact(
        FIGURES_DIR / filename,
        'Figure'
    )

for filename in required_configs:

    verify_artifact(
        RUN_CONFIGS_DIR / filename,
        'Configuration'
    )

verify_artifact(MODEL_PATH, 'Model')
verify_artifact(PREDICTIONS_PATH, 'Prediction')

for filename in [
    'training_log.json',
    'training_warnings.json'
]:

    verify_artifact(
        RUN_LOGS_DIR / filename,
        'Training Log'
    )


# Verify shared word vectorizer dependency

word_vectorizer_path = (
    PROJECT_ROOT / '04_models' / 'classical'
    / 'vectorizers' / 'word_tfidf.joblib'
)

word_vectorizer_record = feature_manifest.loc[
    (feature_manifest['Artifact Type'] == 'Vectorizer')
    & (
        feature_manifest['Path']
        == str(word_vectorizer_path.relative_to(PROJECT_ROOT))
    )
]

assert len(word_vectorizer_record) == 1

assert calculate_sha256(word_vectorizer_path) == (
    word_vectorizer_record.iloc[0]['SHA256']
)

verify_artifact(
    word_vectorizer_path,
    'Shared Dependency'
)

verify_artifact(
    SRC_DIR / 'evaluation_utils.py',
    'Shared Dependency'
)

verify_artifact(
    FEATURES_DIR / 'feature_artifact_manifest.csv',
    'Shared Dependency'
)

assert calculate_sha256(
    SRC_DIR / 'evaluation_utils.py'
) == evaluation_hash

assert calculate_sha256(
    FEATURES_DIR / 'feature_artifact_manifest.csv'
) == feature_manifest_hash


# Verify all required artifacts are available

final_verification_df = pd.DataFrame(
    verification_records
)

assert final_verification_df['Path'].is_unique

final_verification_df.to_csv(
    TABLES_DIR / 'dt_final_artifact_verification.csv',
    index=False
)

missing_artifacts_df = final_verification_df.loc[
    final_verification_df['Status'] == 'Missing'
]

if not missing_artifacts_df.empty:
    display(missing_artifacts_df)

assert missing_artifacts_df.empty

print("\nFinal artifact verification")

display(
    final_verification_df
    .groupby(['Type', 'Status'])
    .size()
    .reset_index(name='Artifacts')
)


# Verify the completed analysis configuration

with open(DT_ANALYSIS_CONFIG_PATH, 'r') as file:
    saved_analysis_config = json.load(file)

assert saved_analysis_config['run_id'] == RUN_ID
assert saved_analysis_config['common_baseline_rank'] == dt_rank
assert saved_analysis_config['retained_as_nonlinear_baseline']
assert not saved_analysis_config['official_test_used']
assert not saved_analysis_config['final_project_model_frozen']


# Construct the shared experiment registry record

registry_record = {
    'run_id': RUN_ID,
    'notebook': NOTEBOOK_DIR.name,
    'model': MODEL_NAME,
    'feature_representation': REPRESENTATION,
    'loss_function': 'Gini impurity (split criterion)',
    'sampling_strategy': 'representative_multilabel',
    'training_samples': len(y_train),
    'seed': SEED,
    'max_length': '',
    'config_path': str(
        RUN_CONFIG_PATH.relative_to(PROJECT_ROOT)
    ),
    'model_path': str(
        MODEL_PATH.relative_to(PROJECT_ROOT)
    ),
    'results_path': str(
        RUN_DIR.relative_to(PROJECT_ROOT)
    ),
    'validation_macro_ap': run_summary['Macro AP'],
    'validation_macro_f1':
        run_summary['Optimized Macro F1'],
    'training_time_seconds': training_time,
    'status': 'Completed',
    'start_time': training_start_time,
    'end_time': training_end_time,
    'notes': (
        'Resource-controlled nonlinear baseline. '
        'Gini criterion, maximum depth 30, '
        'maximum 512 leaves, minimum leaf size 5. '
        'Unweighted One-vs-Rest classification. '
        'Validation-optimized per-label thresholds. '
        'Training time includes CSC conversion. '
        'Substantial training-validation gap observed. '
        'One datetime deprecation warning retained in logs. '
        'Official test not used.'
    )
}


# Preserve previously registered experiments

registry_before_df = pd.read_csv(
    REGISTRY_PATH,
    dtype=str,
    keep_default_na=False
)

assert registry_before_df['run_id'].is_unique

assert set(registry_record) == set(
    registry_before_df.columns
)

previous_registry_df = registry_before_df.loc[
    registry_before_df['run_id'] != RUN_ID
].copy()

assert set(required_previous_runs).issubset(
    set(previous_registry_df['run_id'])
)

new_record_df = pd.DataFrame(
    [registry_record],
    columns=registry_before_df.columns
).astype(str)

updated_registry_df = pd.concat(
    [previous_registry_df, new_record_df],
    ignore_index=True
)


# Verify registry integrity before saving

def verify_dt_registry(candidate_df):

    assert candidate_df['run_id'].is_unique

    assert len(candidate_df) == (
        len(previous_registry_df) + 1
    )

    preserved_records = candidate_df.loc[
        candidate_df['run_id'] != RUN_ID
    ]

    pd.testing.assert_frame_equal(
        previous_registry_df.reset_index(drop=True),
        preserved_records.reset_index(drop=True),
        check_exact=True
    )

    record = candidate_df.loc[
        candidate_df['run_id'] == RUN_ID
    ].iloc[0]

    assert record['status'] == 'Completed'
    assert record['model'] == MODEL_NAME

    assert np.isclose(
        float(record['validation_macro_ap']),
        run_summary['Macro AP'],
        rtol=0,
        atol=1e-12
    )

    assert np.isclose(
        float(record['validation_macro_f1']),
        run_summary['Optimized Macro F1'],
        rtol=0,
        atol=1e-12
    )

    assert np.isclose(
        float(record['training_time_seconds']),
        training_time,
        rtol=0,
        atol=1e-12
    )

    assert int(record['training_samples']) == len(y_train)
    assert int(record['seed']) == SEED

    assert (PROJECT_ROOT / record['model_path']).is_file()
    assert (PROJECT_ROOT / record['config_path']).is_file()
    assert (PROJECT_ROOT / record['results_path']).is_dir()


temporary_registry_path = REGISTRY_PATH.with_suffix('.tmp')

updated_registry_df.to_csv(
    temporary_registry_path,
    index=False
)

temporary_registry_df = pd.read_csv(
    temporary_registry_path,
    dtype=str,
    keep_default_na=False
)

verify_dt_registry(temporary_registry_df)

temporary_registry_path.replace(REGISTRY_PATH)

saved_registry_df = pd.read_csv(
    REGISTRY_PATH,
    dtype=str,
    keep_default_na=False
)

verify_dt_registry(saved_registry_df)


# Generate the updated validation leaderboard

leaderboard_columns = [
    'run_id',
    'model',
    'feature_representation',
    'training_samples',
    'validation_macro_ap',
    'validation_macro_f1',
    'training_time_seconds',
    'status'
]

leaderboard_df = saved_registry_df.loc[
    saved_registry_df['status'] == 'Completed',
    leaderboard_columns
].copy()

for column in [
    'training_samples',
    'validation_macro_ap',
    'validation_macro_f1',
    'training_time_seconds'
]:

    leaderboard_df[column] = pd.to_numeric(
        leaderboard_df[column]
    )

leaderboard_df = (
    leaderboard_df.sort_values(
        [
            'validation_macro_ap',
            'validation_macro_f1',
            'run_id'
        ],
        ascending=[False, False, True]
    )
    .reset_index(drop=True)
)

leaderboard_df.insert(
    0,
    'Rank',
    np.arange(1, len(leaderboard_df) + 1)
)

leaderboard_df.to_csv(
    TABLES_DIR / 'dt_current_validation_leaderboard.csv',
    index=False
)

print("\nUpdated validation leaderboard")

display(leaderboard_df.round(6))


# Record evidence-based methodological decisions

DECISION_LOG_PATH = CONFIG_DIR / 'project_decisions.csv'

decisions_df = pd.read_csv(
    DECISION_LOG_PATH,
    dtype=str,
    keep_default_na=False
)

assert decisions_df['decision_id'].is_unique

dt_decisions = [
    {
        'decision_id': 'D039',
        'component': 'Decision Tree configuration',
        'decision': (
            'Use a complexity-controlled unweighted '
            'DecisionTreeClassifier with word TF-IDF'
        ),
        'justification': (
            'Maintain the common feature comparison while '
            'restricting depth to 30, leaves to 512 and '
            'minimum leaf size to 5. These are predefined '
            'baseline restrictions, not optimized parameters.'
        ),
        'status': 'Implemented'
    },
    {
        'decision_id': 'D040',
        'component': 'Decision Tree structural verification',
        'decision': (
            'Verify tree structure and class alignment '
            'instead of iterative convergence'
        ),
        'justification': (
            'All six trees satisfy predefined structural '
            'restrictions. Every tree reaches the depth '
            'limit, but none reaches the leaf limit. '
            'Model and metric reconstruction passed.'
        ),
        'status': 'Implemented'
    },
    {
        'decision_id': 'D041',
        'component': 'Decision Tree generalization',
        'decision': (
            'Retain training-validation gaps and probability '
            'concentration as model limitation evidence'
        ),
        'justification': (
            f"Training Macro AP is "
            f"{run_summary['Training Macro AP']:.6f}, "
            f"while validation Macro AP is "
            f"{run_summary['Macro AP']:.6f}. "
            f"The resulting gap is "
            f"{run_summary['Macro AP Generalization Gap']:.6f}. "
            'Severe toxicity has the largest AP gap, and '
            'threat has the highest probability concentration.'
        ),
        'status': 'Implemented'
    },
    {
        'decision_id': 'D042',
        'component': 'Decision Tree experiment scope',
        'decision': (
            'Retain one predefined Decision Tree baseline '
            'without additional hyperparameter searches'
        ),
        'justification': (
            f"The evaluated configuration ranks "
            f"{dt_rank} of {len(baseline_results)} "
            'common-feature baselines by validation Macro AP. '
            'Further tuning is deprioritized to preserve '
            'resources for Random Forest and the required '
            'transformer, fairness and explainability stages. '
            'Results do not establish the best achievable '
            'Decision Tree performance.'
        ),
        'status': 'Implemented'
    }
]

assert set(decisions_df.columns) == set(
    dt_decisions[0].keys()
)

dt_decision_ids = [
    record['decision_id']
    for record in dt_decisions
]

previous_decisions_df = decisions_df.loc[
    ~decisions_df['decision_id'].isin(dt_decision_ids)
].copy()

for record in dt_decisions:

    existing = decisions_df.loc[
        decisions_df['decision_id'] == record['decision_id']
    ]

    if not existing.empty:

        assert existing['component'].eq(
            record['component']
        ).all(), "Decision ID collision"

updated_decisions_df = pd.concat(
    [
        previous_decisions_df,
        pd.DataFrame(
            dt_decisions,
            columns=decisions_df.columns
        )
    ],
    ignore_index=True
)

assert updated_decisions_df['decision_id'].is_unique

temporary_decisions_path = (
    DECISION_LOG_PATH.with_suffix('.tmp')
)

updated_decisions_df.to_csv(
    temporary_decisions_path,
    index=False
)

temporary_decisions_df = pd.read_csv(
    temporary_decisions_path,
    dtype=str,
    keep_default_na=False
)

pd.testing.assert_frame_equal(
    updated_decisions_df.reset_index(drop=True),
    temporary_decisions_df.reset_index(drop=True),
    check_exact=True
)

temporary_decisions_path.replace(DECISION_LOG_PATH)

saved_decisions_df = pd.read_csv(
    DECISION_LOG_PATH,
    dtype=str,
    keep_default_na=False
)

pd.testing.assert_frame_equal(
    updated_decisions_df.reset_index(drop=True),
    saved_decisions_df.reset_index(drop=True),
    check_exact=True
)

print("\nNew methodological decisions")

display(
    saved_decisions_df.loc[
        saved_decisions_df['decision_id'].isin(
            dt_decision_ids
        )
    ][
        ['decision_id', 'component', 'decision', 'status']
    ]
)


# Build the final common-feature results table

summary_rows = [
    '| Model | Macro AP | Macro ROC-AUC | Optimized Macro F1 | Optimized Micro F1 |',
    '|---|---:|---:|---:|---:|'
]

for _, row in dt_baseline_ranking_df.iterrows():

    summary_rows.append(
        f"| {row['Model']} "
        f"| {row['Macro AP']:.6f} "
        f"| {row['Macro ROC-AUC']:.6f} "
        f"| {row['Optimized Macro F1']:.6f} "
        f"| {row['Optimized Micro F1']:.6f} |"
    )

summary_table_text = '\n'.join(summary_rows)


# Save the final Decision Tree research summary

summary = f"""# Notebook 03.04 - Decision Tree

## Objective

Evaluate a resource-controlled nonlinear classical
baseline using the common word TF-IDF representation
and identical training and validation partitions.

## Experimental Configuration

- Run ID: {RUN_ID}
- Estimator: DecisionTreeClassifier
- Classification strategy: One-vs-Rest
- Training samples: {len(y_train):,}
- Validation samples: {len(y_val):,}
- Features: {FEATURE_BUDGET:,}
- Criterion: Gini impurity
- Maximum depth: {DT_PARAMS['max_depth']}
- Maximum leaves: {DT_PARAMS['max_leaf_nodes']}
- Minimum samples per leaf: {DT_PARAMS['min_samples_leaf']}
- Minimum samples to split: {DT_PARAMS['min_samples_split']}
- Class weighting: None
- Random seed: {SEED}
- Default threshold: {DEFAULT_THRESHOLD}

## Common Word TF-IDF Comparison

{summary_table_text}

Decision Tree ranks {dt_rank} of {len(baseline_results)}
common-feature configurations by validation Macro AP.

## Generalization

- Training Macro AP: {run_summary['Training Macro AP']:.6f}
- Validation Macro AP: {run_summary['Macro AP']:.6f}
- Macro AP gap: {run_summary['Macro AP Generalization Gap']:.6f}
- Largest per-label AP gap: {largest_gap_row['Label']}
- Largest AP gap: {largest_gap_row['AP Gap']:.6f}

The substantial training-validation gap provides evidence
of overfitting under the evaluated configuration.

All six trees reach the maximum depth restriction.
None reaches the maximum leaf count restriction.

## Probability Resolution

The fitted trees produce a limited number of distinct
probability scores, resulting in extensive score ties.

The largest score concentration occurs for
{highest_concentration_row['Label']}, where
{highest_concentration_row['Most Frequent Score Share (%)']:.6f}%
of validation examples receive the most frequent score.

These ties limit the model's ability to rank comments
within equal-score groups.

## Threshold Optimization

- Default Macro F1: {default_metrics['Macro F1']:.6f}
- Optimized Macro F1: {optimized_metrics['Macro F1']:.6f}
- Default Micro F1: {default_metrics['Micro F1']:.6f}
- Optimized Micro F1: {optimized_metrics['Micro F1']:.6f}

Per-label thresholds improve validation F1 for
{int((per_label_comparison_df['F1 Improvement'] > 0).sum())}
of the six toxicity categories.

Ranking metrics remain unchanged after threshold tuning.

## Structural and Computational Findings

- Binary classifiers: {len(saved_structure_df)}
- Total leaves: {saved_structure_df['Leaf Count'].sum()}
- Total nodes: {saved_structure_df['Node Count'].sum()}
- Maximum observed depth: {saved_structure_df['Actual Depth'].max()}
- Minimum observed leaf size: {saved_structure_df['Minimum Leaf Samples'].min()}
- Model fitting time: {model_fit_time:.6f} seconds
- Total training time: {training_time:.6f} seconds
- Validation prediction time: {validation_prediction_time:.6f} seconds
- Saved model size: {model_size_mib:.6f} MiB

The recorded training time includes sparse CSC conversion.
Saved model size excludes the shared vectorizer.

## Reproducibility

- All six structural checks passed.
- Model and metric reconstruction passed.
- Probability alignment verified.
- Prediction and threshold roundtrips verified.
- Maximum reconstruction error: {maximum_reconstruction_error}
- Critical artifact fingerprint checks: {verified_fingerprint_count}
- Required artifact entries: {len(final_verification_df)}
- Missing artifacts: {len(missing_artifacts_df)}
- Experiment registered successfully.

One datetime deprecation warning was observed and retained
in the training warning log.

## Limitations

- Only one predefined Decision Tree configuration evaluated.
- Hyperparameters were not optimized.
- Results are based on a single validation split and seed.
- Optimized validation F1 may be optimistic.
- Leaf probabilities are not necessarily calibrated.
- Statistical significance has not been established.
- Runtime measurements are from individual runs.
- Saved model size does not represent peak memory usage.
- The generalization results do not establish the best
  achievable performance of Decision Tree models.

## Research Decision

Retain Decision Tree as the planned nonlinear baseline.
Do not conduct additional Decision Tree training at this stage.

The current leading registered configuration is
{leaderboard_df.loc[0, 'run_id']}, selected by validation Macro AP.

Official test data has not been used for model training,
threshold optimization or model selection.

The final project model has not been frozen.

## Next Stage

Notebook 03.05 - Random Forest.
"""

summary_path = RESULTS_DIR / 'summary.md'

summary_path.write_text(
    summary,
    encoding='utf-8'
)

assert summary_path.is_file()
assert summary_path.stat().st_size > 0


# Summarize notebook completion

print("\nDecision Tree completion summary")

print("Notebook-level analysis tables:",
      len(required_tables))

print("Individual experiment tables:",
      len(required_run_tables))

print("Research figures:", len(required_figures))

print("Completed Decision Tree experiments:",
      len(dt_results))

print("Verified binary classifiers:",
      len(saved_structure_df))

print("Critical fingerprint checks:",
      verified_fingerprint_count)

print("Total required artifact entries:",
      len(final_verification_df))

print("Missing artifacts:", len(missing_artifacts_df))

print("Previously registered experiments:",
      len(previous_registry_df))

print("Total registered experiments:",
      len(saved_registry_df))

print("Duplicate run IDs:",
      saved_registry_df['run_id'].duplicated().sum())

print("Total methodological decisions:",
      len(saved_decisions_df))

print("Decision Tree baseline rank:",
      dt_rank, "/", len(baseline_results))

print("Current leading configuration:",
      leaderboard_df.loc[0, 'run_id'])

print("Previous registry records: Preserved")
print("Notebook summary saved.")
print("Official test dataset: Not used")
print("Final project model frozen: No")

print("\nDecision Tree notebook completed successfully.")